In [ ]:
# Fig5 data process
import os
import pandas as pd

# =========================
# 1. File Paths
# =========================
base_dir = "Your path/Stranded_Occupations_Replication/Data/raw_new/oes_wage"
temp_dir = "Your path/Stranded_Occupations_Replication/Data/temp"

sim_path = os.path.join(temp_dir, "simocc_1.xlsx")
wage_path = os.path.join(base_dir, "oes_research_2023_allsectors.xlsx")

out_detail_csv = os.path.join(temp_dir, "sim1_pair_wage_detail.csv")
out_sum_csv = os.path.join(temp_dir, "sim1_pair_wage_sum.csv")

# =========================
# 2. Read Data
# =========================
df_sim = pd.read_excel(sim_path, dtype=str)
df_wage = pd.read_excel(wage_path, dtype=str)

df_sim.columns = [str(c).strip() for c in df_sim.columns]
df_wage.columns = [str(c).strip() for c in df_wage.columns]

# =========================
# 3. Check Required Columns
# =========================
sim_need = ["occupation_code", "occupation_code_match"]
wage_need = ["AREA", "AREA_TITLE", "NAICS", "NAICS_TITLE", "OCC_CODE", "A_MEDIAN"]

miss_sim = [c for c in sim_need if c not in df_sim.columns]
miss_wage = [c for c in wage_need if c not in df_wage.columns]

if miss_sim:
    raise ValueError(f"Missing columns in simocc.xlsx: {miss_sim}")
if miss_wage:
    raise ValueError(f"Missing columns in oes_research_2023_allsectors.xlsx: {miss_wage}")

# =========================
# 4. Standardization Functions
# =========================
def norm_occ(x):
    if pd.isna(x):
        return None
    s = str(x).strip()

    if s.startswith('="') and s.endswith('"'):
        s = s[2:-1]

    s = s.replace('"', "").replace("'", "").replace("=", "")
    s = s.replace("-", "").replace(" ", "").replace("\t", "")

    if s.endswith(".0"):
        s = s[:-2]

    if s == "" or s.lower() == "nan":
        return None

    return s

def norm_naics4(x):
    if pd.isna(x):
        return None
    s = str(x).strip()

    if s.startswith('="') and s.endswith('"'):
        s = s[2:-1]

    s = s.replace('"', "").replace("'", "").replace("=", "")
    s = s.replace("-", "").replace(" ", "").replace("\t", "")

    if s.endswith(".0"):
        s = s[:-2]

    s = "".join(ch for ch in s if ch.isdigit())

    if s == "":
        return None

    return s[:4]

# =========================
# 5. Standardize Variables
# =========================
df_sim["occ_std"] = df_sim["occupation_code"].apply(norm_occ)
df_sim["occ_match_std"] = df_sim["occupation_code_match"].apply(norm_occ)

df_wage["occ_std"] = df_wage["OCC_CODE"].apply(norm_occ)
df_wage["naics4"] = df_wage["NAICS"].apply(norm_naics4)
df_wage["AREA"] = df_wage["AREA"].astype(str).str.strip()
df_wage["AREA_TITLE"] = df_wage["AREA_TITLE"].astype(str).str.strip()

df_wage["A_MEDIAN"] = (
    df_wage["A_MEDIAN"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("$", "", regex=False)
    .str.strip()
)
df_wage["A_MEDIAN"] = pd.to_numeric(df_wage["A_MEDIAN"], errors="coerce")

# Keep only key columns to save memory
df_wage = df_wage[["AREA", "AREA_TITLE", "naics4", "occ_std", "A_MEDIAN"]].copy()

# =========================
# 6. Define Stranded Industries (4-digit NAICS)
# =========================
strand_set = {
    "2111", "2121", "2131", "2211", "2212", "2371",
    "3241", "3251", "3331", "3363", "4235", "4247",
    "4571", "4572", "4861", "4862", "4869"
}

df_wage["is_strand"] = df_wage["naics4"].isin(strand_set).astype(int)

# =========================
# 7. Keep Only Occupations Used in simocc
#    Left occupation: occupation_code
#    Right occupation: occupation_code_match
# =========================
left_occ_set = set(df_sim["occ_std"].dropna().unique())
right_occ_set = set(df_sim["occ_match_std"].dropna().unique())
all_occ_set = left_occ_set | right_occ_set

df_wage = df_wage[df_wage["occ_std"].isin(all_occ_set)].copy()

# =========================
# 8. Calculate Average Wages at State × Occupation Level
#    - strand: average wage in stranded industries
#    - non: average wage in non-stranded industries
# =========================
strand_avg = (
    df_wage[df_wage["is_strand"] == 1]
    .groupby(["AREA", "AREA_TITLE", "occ_std"], as_index=False)["A_MEDIAN"]
    .mean()
    .rename(columns={"A_MEDIAN": "wage_strand_avg"})
)

non_avg = (
    df_wage[df_wage["is_strand"] == 0]
    .groupby(["AREA", "AREA_TITLE", "occ_std"], as_index=False)["A_MEDIAN"]
    .mean()
    .rename(columns={"A_MEDIAN": "wage_non_avg"})
)

# =========================
# 9. Deduplicate simocc to avoid inflation
# =========================
sim_key = df_sim[
    ["occupation_code", "occupation_code_match", "occ_std", "occ_match_std"]
].drop_duplicates().copy()

# =========================
# 10. Match Left: Wages of occupation_code in stranded industries
# =========================
left_panel = sim_key.merge(
    strand_avg,
    left_on="occ_std",
    right_on="occ_std",
    how="left"
)

# =========================
# 11. Match Right: Wages of occupation_code_match in non-stranded industries
# =========================
right_panel = sim_key.merge(
    non_avg,
    left_on="occ_match_std",
    right_on="occ_std",
    how="left"
)

# Rename state variables to avoid column conflicts
right_panel = right_panel.rename(columns={
    "AREA": "AREA_r",
    "AREA_TITLE": "AREA_TITLE_r"
})

# =========================
# 12. Merge left and right by occupation pair + state
#     Get wage comparison for the same state
# =========================
panel = left_panel.merge(
    right_panel[
        [
            "occupation_code",
            "occupation_code_match",
            "AREA_r",
            "AREA_TITLE_r",
            "wage_non_avg"
        ]
    ],
    left_on=["occupation_code", "occupation_code_match", "AREA", "AREA_TITLE"],
    right_on=["occupation_code", "occupation_code_match", "AREA_r", "AREA_TITLE_r"],
    how="outer"
)

# Unify state variables
panel["AREA_final"] = panel["AREA"]
panel["AREA_TITLE_final"] = panel["AREA_TITLE"]

panel["AREA_final"] = panel["AREA_final"].fillna(panel["AREA_r"])
panel["AREA_TITLE_final"] = panel["AREA_TITLE_final"].fillna(panel["AREA_TITLE_r"])

# Clean up columns
drop_cols = [c for c in ["AREA", "AREA_TITLE", "AREA_r", "AREA_TITLE_r"] if c in panel.columns]
panel = panel.drop(columns=drop_cols)

panel = panel.rename(columns={
    "AREA_final": "AREA",
    "AREA_TITLE_final": "AREA_TITLE"
})

# =========================
# 13. Wage Gap Calculation
# =========================
panel["wage_gap"] = panel["wage_non_avg"] - panel["wage_strand_avg"]

# Adjust column order
front_cols = [
    "occupation_code",
    "occupation_code_match",
    "occ_std",
    "occ_match_std",
    "AREA",
    "AREA_TITLE",
    "wage_strand_avg",
    "wage_non_avg",
    "wage_gap"
]
other_cols = [c for c in panel.columns if c not in front_cols]
panel = panel[front_cols + other_cols]

# =========================
# 14. Export Detailed Data
# =========================
panel.to_csv(out_detail_csv, index=False, encoding="utf-8-sig")

# =========================
# 15. Aggregate by Occupation Pair
# =========================
sum_df = (
    panel.groupby(["occupation_code", "occupation_code_match"], as_index=False)
    .agg(
        n_state_strand=("wage_strand_avg", lambda x: x.notna().sum()),
        n_state_non=("wage_non_avg", lambda x: x.notna().sum()),
        n_state_both=("wage_gap", lambda x: x.notna().sum()),
        mean_wage_strand=("wage_strand_avg", "mean"),
        mean_wage_non=("wage_non_avg", "mean"),
        mean_wage_gap=("wage_gap", "mean")
    )
    .sort_values("mean_wage_gap", ascending=False)
    .reset_index(drop=True)
)

sum_df.to_csv(out_sum_csv, index=False, encoding="utf-8-sig")

# =========================
# 16. Print Results
# =========================
print("\n=========== Complete ===========")
print(f"Detailed dataset rows: {len(panel)}")
print(f"Aggregated dataset rows: {len(sum_df)}")

print("\nTop 10 aggregated results:")
print(sum_df.head(10).to_string(index=False))

print("\nOutput files:")
print(out_detail_csv)
print(out_sum_csv)


In [ ]:
import os
import pandas as pd

# =========================
# 1. File Paths
# =========================
base_dir =  "Your path/Stranded_Occupations_Replication/Data/raw_new/oes_wage"
temp_dir = "Your path/Stranded_Occupations_Replication/Data/temp"

sim_path = os.path.join(temp_dir, "simocc_851.xlsx")
wage_path = os.path.join(base_dir, "oes_research_2023_allsectors.xlsx")

out_detail_csv = os.path.join(temp_dir, "sim851_pair_wage_detail.csv")
out_sum_csv = os.path.join(temp_dir, "sim851_pair_wage_sum.csv")

# =========================
# 2. Read Data
# =========================
df_sim = pd.read_excel(sim_path, dtype=str)
df_wage = pd.read_excel(wage_path, dtype=str)

df_sim.columns = [str(c).strip() for c in df_sim.columns]
df_wage.columns = [str(c).strip() for c in df_wage.columns]

# =========================
# 3. Check Required Columns
# =========================
sim_need = ["occupation_code", "occupation_code_match"]
wage_need = ["AREA", "AREA_TITLE", "NAICS", "NAICS_TITLE", "OCC_CODE", "A_MEDIAN"]

miss_sim = [c for c in sim_need if c not in df_sim.columns]
miss_wage = [c for c in wage_need if c not in df_wage.columns]

if miss_sim:
    raise ValueError(f"Missing columns in simocc.xlsx: {miss_sim}")
if miss_wage:
    raise ValueError(f"Missing columns in oes_research_2023_allsectors.xlsx: {miss_wage}")

# =========================
# 4. Standardization Functions
# =========================
def norm_occ(x):
    if pd.isna(x):
        return None
    s = str(x).strip()

    if s.startswith('="') and s.endswith('"'):
        s = s[2:-1]

    s = s.replace('"', "").replace("'", "").replace("=", "")
    s = s.replace("-", "").replace(" ", "").replace("\t", "")

    if s.endswith(".0"):
        s = s[:-2]

    if s == "" or s.lower() == "nan":
        return None

    return s

def norm_naics4(x):
    if pd.isna(x):
        return None
    s = str(x).strip()

    if s.startswith('="') and s.endswith('"'):
        s = s[2:-1]

    s = s.replace('"', "").replace("'", "").replace("=", "")
    s = s.replace("-", "").replace(" ", "").replace("\t", "")

    if s.endswith(".0"):
        s = s[:-2]

    s = "".join(ch for ch in s if ch.isdigit())

    if s == "":
        return None

    return s[:4]

# =========================
# 5. Standardize Variables
# =========================
df_sim["occ_std"] = df_sim["occupation_code"].apply(norm_occ)
df_sim["occ_match_std"] = df_sim["occupation_code_match"].apply(norm_occ)

df_wage["occ_std"] = df_wage["OCC_CODE"].apply(norm_occ)
df_wage["naics4"] = df_wage["NAICS"].apply(norm_naics4)
df_wage["AREA"] = df_wage["AREA"].astype(str).str.strip()
df_wage["AREA_TITLE"] = df_wage["AREA_TITLE"].astype(str).str.strip()

df_wage["A_MEDIAN"] = (
    df_wage["A_MEDIAN"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("$", "", regex=False)
    .str.strip()
)
df_wage["A_MEDIAN"] = pd.to_numeric(df_wage["A_MEDIAN"], errors="coerce")

# Keep only key columns to save memory
df_wage = df_wage[["AREA", "AREA_TITLE", "naics4", "occ_std", "A_MEDIAN"]].copy()

# =========================
# 6. Define Stranded Industries (4-digit NAICS)
# =========================
strand_set = {
    "2111", "2121", "2131", "2211", "2212", "2371",
    "3241", "3251", "3331", "3363", "4235", "4247",
    "4571", "4572", "4861", "4862", "4869"
}

df_wage["is_strand"] = df_wage["naics4"].isin(strand_set).astype(int)

# =========================
# 7. Keep Only Occupations Used in simocc
# =========================
left_occ_set = set(df_sim["occ_std"].dropna().unique())
right_occ_set = set(df_sim["occ_match_std"].dropna().unique())
all_occ_set = left_occ_set | right_occ_set

df_wage = df_wage[df_wage["occ_std"].isin(all_occ_set)].copy()

# =========================
# 8. Calculate Average Wages at State × Occupation Level
# =========================
strand_avg = (
    df_wage[df_wage["is_strand"] == 1]
    .groupby(["AREA", "AREA_TITLE", "occ_std"], as_index=False)["A_MEDIAN"]
    .mean()
    .rename(columns={"A_MEDIAN": "wage_strand_avg"})
)

non_avg = (
    df_wage[df_wage["is_strand"] == 0]
    .groupby(["AREA", "AREA_TITLE", "occ_std"], as_index=False)["A_MEDIAN"]
    .mean()
    .rename(columns={"A_MEDIAN": "wage_non_avg"})
)

# =========================
# 9. Deduplicate simocc to avoid inflation
# =========================
sim_key = df_sim[
    ["occupation_code", "occupation_code_match", "occ_std", "occ_match_std"]
].drop_duplicates().copy()

# =========================
# 10. Match Left: Wages of occupation_code in stranded industries
# =========================
left_panel = sim_key.merge(
    strand_avg,
    left_on="occ_std",
    right_on="occ_std",
    how="left"
)

# =========================
# 11. Match Right: Wages of occupation_code_match in non-stranded industries
# =========================
right_panel = sim_key.merge(
    non_avg,
    left_on="occ_match_std",
    right_on="occ_std",
    how="left"
)

# Rename state variables to avoid column conflicts
right_panel = right_panel.rename(columns={
    "AREA": "AREA_r",
    "AREA_TITLE": "AREA_TITLE_r"
})

# =========================
# 12. Merge left and right by occupation pair + state
# =========================
panel = left_panel.merge(
    right_panel[
        [
            "occupation_code",
            "occupation_code_match",
            "AREA_r",
            "AREA_TITLE_r",
            "wage_non_avg"
        ]
    ],
    left_on=["occupation_code", "occupation_code_match", "AREA", "AREA_TITLE"],
    right_on=["occupation_code", "occupation_code_match", "AREA_r", "AREA_TITLE_r"],
    how="outer"
)

# Unify state variables
panel["AREA_final"] = panel["AREA"]
panel["AREA_TITLE_final"] = panel["AREA_TITLE"]

panel["AREA_final"] = panel["AREA_final"].fillna(panel["AREA_r"])
panel["AREA_TITLE_final"] = panel["AREA_TITLE_final"].fillna(panel["AREA_TITLE_r"])

# Clean up columns
drop_cols = [c for c in ["AREA", "AREA_TITLE", "AREA_r", "AREA_TITLE_r"] if c in panel.columns]
panel = panel.drop(columns=drop_cols)

panel = panel.rename(columns={
    "AREA_final": "AREA",
    "AREA_TITLE_final": "AREA_TITLE"
})

# =========================
# 13. Wage Gap Calculation
# =========================
panel["wage_gap"] = panel["wage_non_avg"] - panel["wage_strand_avg"]

# Adjust column order
front_cols = [
    "occupation_code",
    "occupation_code_match",
    "occ_std",
    "occ_match_std",
    "AREA",
    "AREA_TITLE",
    "wage_strand_avg",
    "wage_non_avg",
    "wage_gap"
]
other_cols = [c for c in panel.columns if c not in front_cols]
panel = panel[front_cols + other_cols]

# =========================
# 14. Export Detailed Data
# =========================
panel.to_csv(out_detail_csv, index=False, encoding="utf-8-sig")

# =========================
# 15. Aggregate by Occupation Pair
# =========================
sum_df = (
    panel.groupby(["occupation_code", "occupation_code_match"], as_index=False)
    .agg(
        n_state_strand=("wage_strand_avg", lambda x: x.notna().sum()),
        n_state_non=("wage_non_avg", lambda x: x.notna().sum()),
        n_state_both=("wage_gap", lambda x: x.notna().sum()),
        mean_wage_strand=("wage_strand_avg", "mean"),
        mean_wage_non=("wage_non_avg", "mean"),
        mean_wage_gap=("wage_gap", "mean")
    )
    .sort_values("mean_wage_gap", ascending=False)
    .reset_index(drop=True)
)

sum_df.to_csv(out_sum_csv, index=False, encoding="utf-8-sig")

# =========================
# 16. Print Results
# =========================
print("\n=========== Complete ===========")
print(f"Detailed dataset rows: {len(panel)}")
print(f"Aggregated dataset rows: {len(sum_df)}")

print("\nTop 10 aggregated results:")
print(sum_df.head(10).to_string(index=False))

print("\nOutput files:")
print(out_detail_csv)
print(out_sum_csv)

In [ ]:
import os
import pandas as pd

# =========================
# 1. File Paths
# =========================
base_dir =  "Your path/Stranded_Occupations_Replication/Data/raw_new/oes_wage"
temp_dir = "Your path/Stranded_Occupations_Replication/Data/temp"

sim_path = os.path.join(temp_dir, "simocc_885.xlsx")
wage_path = os.path.join(base_dir, "oes_research_2023_allsectors.xlsx")

out_detail_csv = os.path.join(temp_dir, "sim885_pair_wage_detail.csv")
out_sum_csv = os.path.join(temp_dir, "sim885_pair_wage_sum.csv")

# =========================
# 2. Read Data
# =========================
df_sim = pd.read_excel(sim_path, dtype=str)
df_wage = pd.read_excel(wage_path, dtype=str)

df_sim.columns = [str(c).strip() for c in df_sim.columns]
df_wage.columns = [str(c).strip() for c in df_wage.columns]

# =========================
# 3. Check Required Columns
# =========================
sim_need = ["occupation_code", "occupation_code_match"]
wage_need = ["AREA", "AREA_TITLE", "NAICS", "NAICS_TITLE", "OCC_CODE", "A_MEDIAN"]

miss_sim = [c for c in sim_need if c not in df_sim.columns]
miss_wage = [c for c in wage_need if c not in df_wage.columns]

if miss_sim:
    raise ValueError(f"Missing columns in simocc_885.xlsx: {miss_sim}")
if miss_wage:
    raise ValueError(f"Missing columns in oes_research_2023_allsectors.xlsx: {miss_wage}")

# =========================
# 4. Standardization Functions
# =========================
def norm_occ(x):
    if pd.isna(x):
        return None
    s = str(x).strip()

    if s.startswith('="') and s.endswith('"'):
        s = s[2:-1]

    s = s.replace('"', "").replace("'", "").replace("=", "")
    s = s.replace("-", "").replace(" ", "").replace("\t", "")

    if s.endswith(".0"):
        s = s[:-2]

    if s == "" or s.lower() == "nan":
        return None

    return s

def norm_naics4(x):
    if pd.isna(x):
        return None
    s = str(x).strip()

    if s.startswith('="') and s.endswith('"'):
        s = s[2:-1]

    s = s.replace('"', "").replace("'", "").replace("=", "")
    s = s.replace("-", "").replace(" ", "").replace("\t", "")

    if s.endswith(".0"):
        s = s[:-2]

    s = "".join(ch for ch in s if ch.isdigit())

    if s == "":
        return None

    return s[:4]

# =========================
# 5. Standardize Variables
# =========================
df_sim["occ_std"] = df_sim["occupation_code"].apply(norm_occ)
df_sim["occ_match_std"] = df_sim["occupation_code_match"].apply(norm_occ)

df_wage["occ_std"] = df_wage["OCC_CODE"].apply(norm_occ)
df_wage["naics4"] = df_wage["NAICS"].apply(norm_naics4)
df_wage["AREA"] = df_wage["AREA"].astype(str).str.strip()
df_wage["AREA_TITLE"] = df_wage["AREA_TITLE"].astype(str).str.strip()

df_wage["A_MEDIAN"] = (
    df_wage["A_MEDIAN"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("$", "", regex=False)
    .str.strip()
)
df_wage["A_MEDIAN"] = pd.to_numeric(df_wage["A_MEDIAN"], errors="coerce")

# Keep only key columns to save memory
df_wage = df_wage[["AREA", "AREA_TITLE", "naics4", "occ_std", "A_MEDIAN"]].copy()

# =========================
# 6. Define Stranded Industries (4-digit NAICS)
# =========================
strand_set = {
    "2111", "2121", "2131", "2211", "2212", "2371",
    "3241", "3251", "3331", "3363", "4235", "4247",
    "4571", "4572", "4861", "4862", "4869"
}

df_wage["is_strand"] = df_wage["naics4"].isin(strand_set).astype(int)

# =========================
# 7. Keep Only Occupations Used in simocc
# =========================
left_occ_set = set(df_sim["occ_std"].dropna().unique())
right_occ_set = set(df_sim["occ_match_std"].dropna().unique())
all_occ_set = left_occ_set | right_occ_set

df_wage = df_wage[df_wage["occ_std"].isin(all_occ_set)].copy()

# =========================
# 8. Calculate Average Wages at State × Occupation Level
# =========================
strand_avg = (
    df_wage[df_wage["is_strand"] == 1]
    .groupby(["AREA", "AREA_TITLE", "occ_std"], as_index=False)["A_MEDIAN"]
    .mean()
    .rename(columns={"A_MEDIAN": "wage_strand_avg"})
)

non_avg = (
    df_wage[df_wage["is_strand"] == 0]
    .groupby(["AREA", "AREA_TITLE", "occ_std"], as_index=False)["A_MEDIAN"]
    .mean()
    .rename(columns={"A_MEDIAN": "wage_non_avg"})
)

# =========================
# 9. Deduplicate simocc to avoid inflation
# =========================
sim_key = df_sim[
    ["occupation_code", "occupation_code_match", "occ_std", "occ_match_std"]
].drop_duplicates().copy()

# =========================
# 10. Match Left: Wages of occupation_code in stranded industries
# =========================
left_panel = sim_key.merge(
    strand_avg,
    left_on="occ_std",
    right_on="occ_std",
    how="left"
)

# =========================
# 11. Match Right: Wages of occupation_code_match in non-stranded industries
# =========================
right_panel = sim_key.merge(
    non_avg,
    left_on="occ_match_std",
    right_on="occ_std",
    how="left"
)

# Rename state variables to avoid column conflicts
right_panel = right_panel.rename(columns={
    "AREA": "AREA_r",
    "AREA_TITLE": "AREA_TITLE_r"
})

# =========================
# 12. Merge left and right by occupation pair + state
# =========================
panel = left_panel.merge(
    right_panel[
        [
            "occupation_code",
            "occupation_code_match",
            "AREA_r",
            "AREA_TITLE_r",
            "wage_non_avg"
        ]
    ],
    left_on=["occupation_code", "occupation_code_match", "AREA", "AREA_TITLE"],
    right_on=["occupation_code", "occupation_code_match", "AREA_r", "AREA_TITLE_r"],
    how="outer"
)

# Unify state variables
panel["AREA_final"] = panel["AREA"]
panel["AREA_TITLE_final"] = panel["AREA_TITLE"]

panel["AREA_final"] = panel["AREA_final"].fillna(panel["AREA_r"])
panel["AREA_TITLE_final"] = panel["AREA_TITLE_final"].fillna(panel["AREA_TITLE_r"])

# Clean up columns
drop_cols = [c for c in ["AREA", "AREA_TITLE", "AREA_r", "AREA_TITLE_r"] if c in panel.columns]
panel = panel.drop(columns=drop_cols)

panel = panel.rename(columns={
    "AREA_final": "AREA",
    "AREA_TITLE_final": "AREA_TITLE"
})

# =========================
# 13. Wage Gap Calculation
# =========================
panel["wage_gap"] = panel["wage_non_avg"] - panel["wage_strand_avg"]

# Adjust column order
front_cols = [
    "occupation_code",
    "occupation_code_match",
    "occ_std",
    "occ_match_std",
    "AREA",
    "AREA_TITLE",
    "wage_strand_avg",
    "wage_non_avg",
    "wage_gap"
]
other_cols = [c for c in panel.columns if c not in front_cols]
panel = panel[front_cols + other_cols]

# =========================
# 14. Export Detailed Data
# =========================
panel.to_csv(out_detail_csv, index=False, encoding="utf-8-sig")

# =========================
# 15. Aggregate by Occupation Pair
# =========================
sum_df = (
    panel.groupby(["occupation_code", "occupation_code_match"], as_index=False)
    .agg(
        n_state_strand=("wage_strand_avg", lambda x: x.notna().sum()),
        n_state_non=("wage_non_avg", lambda x: x.notna().sum()),
        n_state_both=("wage_gap", lambda x: x.notna().sum()),
        mean_wage_strand=("wage_strand_avg", "mean"),
        mean_wage_non=("wage_non_avg", "mean"),
        mean_wage_gap=("wage_gap", "mean")
    )
    .sort_values("mean_wage_gap", ascending=False)
    .reset_index(drop=True)
)

sum_df.to_csv(out_sum_csv, index=False, encoding="utf-8-sig")

# =========================
# 16. Print Results
# =========================
print("\n=========== Complete ===========")
print(f"Detailed dataset rows: {len(panel)}")
print(f"Aggregated dataset rows: {len(sum_df)}")

print("\nTop 10 aggregated results:")
print(sum_df.head(10).to_string(index=False))

print("\nOutput files:")
print(out_detail_csv)
print(out_sum_csv)


In [ ]:
# 2014 Wage Data Processing (sim1 / sim885 / sim851)
import os
import pandas as pd

# =========================
# 1. File Paths
# =========================
base_dir =  "Your path/Stranded_Occupations_Replication/Data/raw_new/oes_wage"
temp_dir = "Your path/Stranded_Occupations_Replication/Data/temp"

wage_path = os.path.join(base_dir, "oes_research_2014_allsectors.xlsx")

# Three sim files and corresponding outputs
tasks = [
    {
        "sim_path":      os.path.join(temp_dir, "simocc_1.xlsx"),
        "out_detail":    os.path.join(temp_dir, "sim1_pair_wage_detail_14.csv"),
        "out_sum":       os.path.join(temp_dir, "sim1_pair_wage_sum_14.csv"),
        "label":         "sim1",
    },
    {
        "sim_path":      os.path.join(temp_dir, "simocc_885.xlsx"),
        "out_detail":    os.path.join(temp_dir, "sim885_pair_wage_detail_14.csv"),
        "out_sum":       os.path.join(temp_dir, "sim885_pair_wage_sum_14.csv"),
        "label":         "sim885",
    },
    {
        "sim_path":      os.path.join(temp_dir, "simocc_851.xlsx"),
        "out_detail":    os.path.join(temp_dir, "sim851_pair_wage_detail_14.csv"),
        "out_sum":       os.path.join(temp_dir, "sim851_pair_wage_sum_14.csv"),
        "label":         "sim851",
    },
]

# =========================
# 2. Standardization Functions
# =========================
def norm_occ(x):
    """Standardize occupation codes: remove special characters, hyphens, spaces"""
    if pd.isna(x):
        return None
    s = str(x).strip()
    if s.startswith('="') and s.endswith('"'):
        s = s[2:-1]
    s = s.replace('"', "").replace("'", "").replace("=", "")
    s = s.replace("-", "").replace(" ", "").replace("\t", "")
    if s.endswith(".0"):
        s = s[:-2]
    if s == "" or s.lower() == "nan":
        return None
    return s

def norm_naics4(x):
    """Standardize NAICS codes: keep digits only, take first 4 digits"""
    if pd.isna(x):
        return None
    s = str(x).strip()
    if s.startswith('="') and s.endswith('"'):
        s = s[2:-1]
    s = s.replace('"', "").replace("'", "").replace("=", "")
    s = s.replace("-", "").replace(" ", "").replace("\t", "")
    if s.endswith(".0"):
        s = s[:-2]
    s = "".join(ch for ch in s if ch.isdigit())
    if s == "":
        return None
    return s[:4]

# =========================
# 3. Stranded Industries Definition (4-digit NAICS)
# =========================
strand_set = {
    "2111", "2121", "2131", "2211", "2212", "2371",
    "3241", "3251", "3331", "3363", "4235", "4247",
    "4571", "4572", "4861", "4862", "4869"
}

# =========================
# 4. Read and Preprocess Wage Data (Read once only)
#    2014 variable names: all lowercase, occupation code column = "occ code" (with space)
# =========================
print("Reading wage data...")
df_wage_raw = pd.read_excel(wage_path, dtype=str)

# Standardize column names: strip spaces, convert to lowercase
df_wage_raw.columns = [str(c).strip().lower() for c in df_wage_raw.columns]

# Check required columns (adapted for 2014 naming)
wage_need = ["area", "area_title", "naics", "naics_title", "occ code", "a_median"]
miss_wage = [c for c in wage_need if c not in df_wage_raw.columns]
if miss_wage:
    raise ValueError(f"Missing columns in oes_research_2014_allsectors.xlsx: {miss_wage}")

# Standardize key columns
df_wage_raw["occ_std"]  = df_wage_raw["occ code"].apply(norm_occ)   # Note: 2014 uses "occ code"
df_wage_raw["naics4"]   = df_wage_raw["naics"].apply(norm_naics4)
df_wage_raw["area"]      = df_wage_raw["area"].astype(str).str.strip()
df_wage_raw["area_title"]= df_wage_raw["area_title"].astype(str).str.strip()

df_wage_raw["a_median"] = (
    df_wage_raw["a_median"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("$", "", regex=False)
    .str.strip()
)
df_wage_raw["a_median"] = pd.to_numeric(df_wage_raw["a_median"], errors="coerce")

# Keep only columns needed for subsequent analysis
df_wage_base = df_wage_raw[
    ["area", "area_title", "naics4", "occ_std", "a_median"]
].copy()

# Flag stranded industries
df_wage_base["is_strand"] = df_wage_base["naics4"].isin(strand_set).astype(int)

print(f"Wage data loaded successfully, total rows: {len(df_wage_base):,}\n")

# =========================
# 5. Process each sim file in loop
# =========================
for task in tasks:
    label      = task["label"]
    sim_path   = task["sim_path"]
    out_detail = task["out_detail"]
    out_sum    = task["out_sum"]

    print(f"========== Processing {label} ==========")

    # --- 5.1 Read sim file ---
    df_sim = pd.read_excel(sim_path, dtype=str)
    df_sim.columns = [str(c).strip() for c in df_sim.columns]

    sim_need = ["occupation_code", "occupation_code_match"]
    miss_sim = [c for c in sim_need if c not in df_sim.columns]
    if miss_sim:
        raise ValueError(f"Missing columns in {sim_path}: {miss_sim}")

    df_sim["occ_std"]       = df_sim["occupation_code"].apply(norm_occ)
    df_sim["occ_match_std"] = df_sim["occupation_code_match"].apply(norm_occ)

    # --- 5.2 Keep only occupations used in this sim to save memory ---
    left_occ_set  = set(df_sim["occ_std"].dropna().unique())
    right_occ_set = set(df_sim["occ_match_std"].dropna().unique())
    all_occ_set   = left_occ_set | right_occ_set

    df_wage = df_wage_base[df_wage_base["occ_std"].isin(all_occ_set)].copy()

    # --- 5.3 Calculate average wages at State × Occupation level ---
    # Stranded industries
    strand_avg = (
        df_wage[df_wage["is_strand"] == 1]
        .groupby(["area", "area_title", "occ_std"], as_index=False)["a_median"]
        .mean()
        .rename(columns={"a_median": "wage_strand_avg"})
    )

    # Non-stranded industries
    non_avg = (
        df_wage[df_wage["is_strand"] == 0]
        .groupby(["area", "area_title", "occ_std"], as_index=False)["a_median"]
        .mean()
        .rename(columns={"a_median": "wage_non_avg"})
    )

    # --- 5.4 Deduplicate sim data ---
    sim_key = df_sim[
        ["occupation_code", "occupation_code_match", "occ_std", "occ_match_std"]
    ].drop_duplicates().copy()

    # --- 5.5 Left side: match occupation_code with stranded industry wages ---
    left_panel = sim_key.merge(
        strand_avg,
        on="occ_std",
        how="left"
    )

    # --- 5.6 Right side: match occupation_code_match with non-stranded industry wages ---
    right_panel = sim_key.merge(
        non_avg,
        left_on="occ_match_std",
        right_on="occ_std",
        how="left"
    ).rename(columns={
        "area":       "area_r",
        "area_title": "area_title_r"
    })

    # --- 5.7 Merge left and right by occupation pair + state ---
    panel = left_panel.merge(
        right_panel[[
            "occupation_code", "occupation_code_match",
            "area_r", "area_title_r", "wage_non_avg"
        ]],
        left_on=["occupation_code", "occupation_code_match", "area", "area_title"],
        right_on=["occupation_code", "occupation_code_match", "area_r", "area_title_r"],
        how="outer"
    )

    # Unify state variables
    panel["area_final"]       = panel["area"].fillna(panel["area_r"])
    panel["area_title_final"] = panel["area_title"].fillna(panel["area_title_r"])

    drop_cols = [c for c in ["area", "area_title", "area_r", "area_title_r"] if c in panel.columns]
    panel = panel.drop(columns=drop_cols).rename(columns={
        "area_final":       "area",
        "area_title_final": "area_title"
    })

    # --- 5.8 Calculate wage gap ---
    panel["wage_gap"] = panel["wage_non_avg"] - panel["wage_strand_avg"]

    # Reorder columns
    front_cols = [
        "occupation_code", "occupation_code_match",
        "occ_std", "occ_match_std",
        "area", "area_title",
        "wage_strand_avg", "wage_non_avg", "wage_gap"
    ]
    other_cols = [c for c in panel.columns if c not in front_cols]
    panel = panel[front_cols + other_cols]

    # --- 5.9 Export detailed data ---
    panel.to_csv(out_detail, index=False, encoding="utf-8-sig")

    # --- 5.10 Aggregate by occupation pair ---
    sum_df = (
        panel.groupby(["occupation_code", "occupation_code_match"], as_index=False)
        .agg(
            n_state_strand=("wage_strand_avg", lambda x: x.notna().sum()),
            n_state_non   =("wage_non_avg",    lambda x: x.notna().sum()),
            n_state_both  =("wage_gap",        lambda x: x.notna().sum()),
            mean_wage_strand=("wage_strand_avg", "mean"),
            mean_wage_non   =("wage_non_avg",    "mean"),
            mean_wage_gap   =("wage_gap",        "mean")
        )
        .sort_values("mean_wage_gap", ascending=False)
        .reset_index(drop=True)
    )

    sum_df.to_csv(out_sum, index=False, encoding="utf-8-sig")

    # --- 5.11 Print results ---
    print(f"Detailed dataset rows: {len(panel):,}")
    print(f"Aggregated dataset rows: {len(sum_df):,}")
    print("\nTop 10 aggregated results:")
    print(sum_df.head(10).to_string(index=False))
    print(f"\nOutput files:\n  {out_detail}\n  {out_sum}\n")

print("=========== All tasks completed ===========")


In [ ]:
# 2017 Wage Data Processing (sim1 / sim885 / sim851)
import os
import pandas as pd

# =========================
# 1. File Paths
# =========================
base_dir =  "Your path/Stranded_Occupations_Replication/Data/raw_new/oes_wage"
temp_dir = "Your path/Stranded_Occupations_Replication/Data/temp"

wage_path = os.path.join(base_dir, "oes_research_2017_allsectors.xlsx")

# Three sim files and corresponding outputs
tasks = [
    {
        "sim_path":      os.path.join(temp_dir, "simocc_1.xlsx"),
        "out_detail":    os.path.join(temp_dir, "sim1_pair_wage_detail_17.csv"),
        "out_sum":       os.path.join(temp_dir, "sim1_pair_wage_sum_17.csv"),
        "label":         "sim1",
    },
    {
        "sim_path":      os.path.join(temp_dir, "simocc_885.xlsx"),
        "out_detail":    os.path.join(temp_dir, "sim885_pair_wage_detail_17.csv"),
        "out_sum":       os.path.join(temp_dir, "sim885_pair_wage_sum_17.csv"),
        "label":         "sim885",
    },
    {
        "sim_path":      os.path.join(temp_dir, "simocc_851.xlsx"),
        "out_detail":    os.path.join(temp_dir, "sim851_pair_wage_detail_17.csv"),
        "out_sum":       os.path.join(temp_dir, "sim851_pair_wage_sum_17.csv"),
        "label":         "sim851",
    },
]

# =========================
# 2. Standardization Functions
# =========================
def norm_occ(x):
    """Standardize occupation codes: remove special characters, hyphens, spaces"""
    if pd.isna(x):
        return None
    s = str(x).strip()
    if s.startswith('="') and s.endswith('"'):
        s = s[2:-1]
    s = s.replace('"', "").replace("'", "").replace("=", "")
    s = s.replace("-", "").replace(" ", "").replace("\t", "")
    if s.endswith(".0"):
        s = s[:-2]
    if s == "" or s.lower() == "nan":
        return None
    return s

def norm_naics4(x):
    """Standardize NAICS codes: keep digits only, take first 4 digits"""
    if pd.isna(x):
        return None
    s = str(x).strip()
    if s.startswith('="') and s.endswith('"'):
        s = s[2:-1]
    s = s.replace('"', "").replace("'", "").replace("=", "")
    s = s.replace("-", "").replace(" ", "").replace("\t", "")
    if s.endswith(".0"):
        s = s[:-2]
    s = "".join(ch for ch in s if ch.isdigit())
    if s == "":
        return None
    return s[:4]

# =========================
# 3. Stranded Industries Definition (4-digit NAICS)
# =========================
strand_set = {
    "2111", "2121", "2131", "2211", "2212", "2371",
    "3241", "3251", "3331", "3363", "4235", "4247",
    "4571", "4572", "4861", "4862", "4869"
}

# =========================
# 4. Read and Preprocess Wage Data (Read once only)
#    2017 variable names: all lowercase, occupation code column = "occ code" (with space)
# =========================
print("Reading wage data...")
df_wage_raw = pd.read_excel(wage_path, dtype=str)

# Standardize column names: strip spaces, convert to lowercase
df_wage_raw.columns = [str(c).strip().lower() for c in df_wage_raw.columns]

# Check required columns (adapted for 2017 naming)
wage_need = ["area", "area_title", "naics", "naics_title", "occ code", "a_median"]
miss_wage = [c for c in wage_need if c not in df_wage_raw.columns]
if miss_wage:
    raise ValueError(f"Missing columns in oes_research_2017_allsectors.xlsx: {miss_wage}")

# Standardize key columns
df_wage_raw["occ_std"]  = df_wage_raw["occ code"].apply(norm_occ)
df_wage_raw["naics4"]   = df_wage_raw["naics"].apply(norm_naics4)
df_wage_raw["area"]      = df_wage_raw["area"].astype(str).str.strip()
df_wage_raw["area_title"]= df_wage_raw["area_title"].astype(str).str.strip()

df_wage_raw["a_median"] = (
    df_wage_raw["a_median"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("$", "", regex=False)
    .str.strip()
)
df_wage_raw["a_median"] = pd.to_numeric(df_wage_raw["a_median"], errors="coerce")

# Keep only columns needed for subsequent analysis
df_wage_base = df_wage_raw[
    ["area", "area_title", "naics4", "occ_std", "a_median"]
].copy()

# Flag stranded industries
df_wage_base["is_strand"] = df_wage_base["naics4"].isin(strand_set).astype(int)

print(f"Wage data loaded successfully, total rows: {len(df_wage_base):,}\n")

# =========================
# 5. Process each sim file in loop
# =========================
for task in tasks:
    label      = task["label"]
    sim_path   = task["sim_path"]
    out_detail = task["out_detail"]
    out_sum    = task["out_sum"]

    print(f"========== Processing {label} ==========")

    # --- 5.1 Read sim file ---
    df_sim = pd.read_excel(sim_path, dtype=str)
    df_sim.columns = [str(c).strip() for c in df_sim.columns]

    sim_need = ["occupation_code", "occupation_code_match"]
    miss_sim = [c for c in sim_need if c not in df_sim.columns]
    if miss_sim:
        raise ValueError(f"Missing columns in {sim_path}: {miss_sim}")

    df_sim["occ_std"]       = df_sim["occupation_code"].apply(norm_occ)
    df_sim["occ_match_std"] = df_sim["occupation_code_match"].apply(norm_occ)

    # --- 5.2 Keep only occupations used in this sim to save memory ---
    left_occ_set  = set(df_sim["occ_std"].dropna().unique())
    right_occ_set = set(df_sim["occ_match_std"].dropna().unique())
    all_occ_set   = left_occ_set | right_occ_set

    df_wage = df_wage_base[df_wage_base["occ_std"].isin(all_occ_set)].copy()

    # --- 5.3 Calculate average wages at State × Occupation level ---
    # Stranded industries
    strand_avg = (
        df_wage[df_wage["is_strand"] == 1]
        .groupby(["area", "area_title", "occ_std"], as_index=False)["a_median"]
        .mean()
        .rename(columns={"a_median": "wage_strand_avg"})
    )

    # Non-stranded industries
    non_avg = (
        df_wage[df_wage["is_strand"] == 0]
        .groupby(["area", "area_title", "occ_std"], as_index=False)["a_median"]
        .mean()
        .rename(columns={"a_median": "wage_non_avg"})
    )

    # --- 5.4 Deduplicate sim data ---
    sim_key = df_sim[
        ["occupation_code", "occupation_code_match", "occ_std", "occ_match_std"]
    ].drop_duplicates().copy()

    # --- 5.5 Left side: match occupation_code with stranded industry wages ---
    left_panel = sim_key.merge(
        strand_avg,
        on="occ_std",
        how="left"
    )

    # --- 5.6 Right side: match occupation_code_match with non-stranded industry wages ---
    right_panel = sim_key.merge(
        non_avg,
        left_on="occ_match_std",
        right_on="occ_std",
        how="left"
    ).rename(columns={
        "area":       "area_r",
        "area_title": "area_title_r"
    })

    # --- 5.7 Merge left and right by occupation pair + state ---
    panel = left_panel.merge(
        right_panel[[
            "occupation_code", "occupation_code_match",
            "area_r", "area_title_r", "wage_non_avg"
        ]],
        left_on=["occupation_code", "occupation_code_match", "area", "area_title"],
        right_on=["occupation_code", "occupation_code_match", "area_r", "area_title_r"],
        how="outer"
    )

    # Unify state variables
    panel["area_final"]       = panel["area"].fillna(panel["area_r"])
    panel["area_title_final"] = panel["area_title"].fillna(panel["area_title_r"])

    drop_cols = [c for c in ["area", "area_title", "area_r", "area_title_r"] if c in panel.columns]
    panel = panel.drop(columns=drop_cols).rename(columns={
        "area_final":       "area",
        "area_title_final": "area_title"
    })

    # --- 5.8 Calculate wage gap ---
    panel["wage_gap"] = panel["wage_non_avg"] - panel["wage_strand_avg"]

    # Reorder columns
    front_cols = [
        "occupation_code", "occupation_code_match",
        "occ_std", "occ_match_std",
        "area", "area_title",
        "wage_strand_avg", "wage_non_avg", "wage_gap"
    ]
    other_cols = [c for c in panel.columns if c not in front_cols]
    panel = panel[front_cols + other_cols]

    # --- 5.9 Export detailed data ---
    panel.to_csv(out_detail, index=False, encoding="utf-8-sig")

    # --- 5.10 Aggregate by occupation pair ---
    sum_df = (
        panel.groupby(["occupation_code", "occupation_code_match"], as_index=False)
        .agg(
            n_state_strand=("wage_strand_avg", lambda x: x.notna().sum()),
            n_state_non   =("wage_non_avg",    lambda x: x.notna().sum()),
            n_state_both  =("wage_gap",        lambda x: x.notna().sum()),
            mean_wage_strand=("wage_strand_avg", "mean"),
            mean_wage_non   =("wage_non_avg",    "mean"),
            mean_wage_gap   =("wage_gap",        "mean")
        )
        .sort_values("mean_wage_gap", ascending=False)
        .reset_index(drop=True)
    )

    sum_df.to_csv(out_sum, index=False, encoding="utf-8-sig")

    # --- 5.11 Print results ---
    print(f"Detailed dataset rows: {len(panel):,}")
    print(f"Aggregated dataset rows: {len(sum_df):,}")
    print("\nTop 10 aggregated results:")
    print(sum_df.head(10).to_string(index=False))
    print(f"\nOutput files:\n  {out_detail}\n  {out_sum}\n")

print("=========== All tasks completed ===========")


In [ ]:
# 2020 Wage Data Processing (sim1 / sim885 / sim851)
import os
import pandas as pd

# =========================
# 1. File Paths
# =========================
base_dir =  "Your path/Stranded_Occupations_Replication/Data/raw_new/oes_wage"
temp_dir = "Your path/Stranded_Occupations_Replication/Data/temp"

wage_path = os.path.join(base_dir, "oes_research_2020_allsectors.xlsx")

# Three sim files and corresponding outputs
tasks = [
    {
        "sim_path":   os.path.join(temp_dir, "simocc_1.xlsx"),
        "out_detail": os.path.join(temp_dir, "sim1_pair_wage_detail_20.csv"),
        "out_sum":    os.path.join(temp_dir, "sim1_pair_wage_sum_20.csv"),
        "label":      "sim1",
    },
    {
        "sim_path":   os.path.join(temp_dir, "simocc_885.xlsx"),
        "out_detail": os.path.join(temp_dir, "sim885_pair_wage_detail_20.csv"),
        "out_sum":    os.path.join(temp_dir, "sim885_pair_wage_sum_20.csv"),
        "label":      "sim885",
    },
    {
        "sim_path":   os.path.join(temp_dir, "simocc_851.xlsx"),
        "out_detail": os.path.join(temp_dir, "sim851_pair_wage_detail_20.csv"),
        "out_sum":    os.path.join(temp_dir, "sim851_pair_wage_sum_20.csv"),
        "label":      "sim851",
    },
]

# =========================
# 2. Standardization Functions
# =========================
def norm_occ(x):
    """Standardize occupation codes: remove special characters, hyphens, spaces"""
    if pd.isna(x):
        return None
    s = str(x).strip()
    if s.startswith('="') and s.endswith('"'):
        s = s[2:-1]
    s = s.replace('"', "").replace("'", "").replace("=", "")
    s = s.replace("-", "").replace(" ", "").replace("\t", "")
    if s.endswith(".0"):
        s = s[:-2]
    if s == "" or s.lower() == "nan":
        return None
    return s

def norm_naics4(x):
    """Standardize NAICS codes: keep digits only, take first 4 digits"""
    if pd.isna(x):
        return None
    s = str(x).strip()
    if s.startswith('="') and s.endswith('"'):
        s = s[2:-1]
    s = s.replace('"', "").replace("'", "").replace("=", "")
    s = s.replace("-", "").replace(" ", "").replace("\t", "")
    if s.endswith(".0"):
        s = s[:-2]
    s = "".join(ch for ch in s if ch.isdigit())
    if s == "":
        return None
    return s[:4]

# =========================
# 3. Stranded Industries Definition (4-digit NAICS)
# =========================
strand_set = {
    "2111", "2121", "2131", "2211", "2212", "2371",
    "3241", "3251", "3331", "3363", "4235", "4247",
    "4571", "4572", "4861", "4862", "4869"
}

# =========================
# 4. Read and Preprocess Wage Data (Read once only)
#    2020 variable names: UPPERCASE, occupation code column = "OCC_CODE" (no space)
# =========================
print("Reading wage data...")
df_wage_raw = pd.read_excel(wage_path, dtype=str)

# Standardize column names: strip spaces, convert to lowercase
df_wage_raw.columns = [str(c).strip().lower() for c in df_wage_raw.columns]

# Check required columns (after converting to lowercase)
wage_need = ["area", "area_title", "naics", "naics_title", "occ_code", "a_median"]
miss_wage = [c for c in wage_need if c not in df_wage_raw.columns]
if miss_wage:
    raise ValueError(f"Missing columns in oes_research_2020_allsectors.xlsx: {miss_wage}")

# Standardize key columns
df_wage_raw["occ_std"]   = df_wage_raw["occ_code"].apply(norm_occ)
df_wage_raw["naics4"]    = df_wage_raw["naics"].apply(norm_naics4)
df_wage_raw["area"]      = df_wage_raw["area"].astype(str).str.strip()
df_wage_raw["area_title"]= df_wage_raw["area_title"].astype(str).str.strip()

df_wage_raw["a_median"] = (
    df_wage_raw["a_median"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("$", "", regex=False)
    .str.strip()
)
df_wage_raw["a_median"] = pd.to_numeric(df_wage_raw["a_median"], errors="coerce")

# Keep only columns needed for subsequent analysis
df_wage_base = df_wage_raw[
    ["area", "area_title", "naics4", "occ_std", "a_median"]
].copy()

# Flag stranded industries
df_wage_base["is_strand"] = df_wage_base["naics4"].isin(strand_set).astype(int)

print(f"Wage data loaded successfully, total rows: {len(df_wage_base):,}\n")

# =========================
# 5. Process each sim file in loop
# =========================
for task in tasks:
    label      = task["label"]
    sim_path   = task["sim_path"]
    out_detail = task["out_detail"]
    out_sum    = task["out_sum"]

    print(f"========== Processing {label} ==========")

    # --- 5.1 Read sim file ---
    df_sim = pd.read_excel(sim_path, dtype=str)
    df_sim.columns = [str(c).strip() for c in df_sim.columns]

    sim_need = ["occupation_code", "occupation_code_match"]
    miss_sim = [c for c in sim_need if c not in df_sim.columns]
    if miss_sim:
        raise ValueError(f"Missing columns in {sim_path}: {miss_sim}")

    df_sim["occ_std"]       = df_sim["occupation_code"].apply(norm_occ)
    df_sim["occ_match_std"] = df_sim["occupation_code_match"].apply(norm_occ)

    # --- 5.2 Keep only occupations used in this sim to save memory ---
    left_occ_set  = set(df_sim["occ_std"].dropna().unique())
    right_occ_set = set(df_sim["occ_match_std"].dropna().unique())
    all_occ_set   = left_occ_set | right_occ_set

    df_wage = df_wage_base[df_wage_base["occ_std"].isin(all_occ_set)].copy()

    # --- 5.3 Calculate average wages at State × Occupation level ---
    # Stranded industries
    strand_avg = (
        df_wage[df_wage["is_strand"] == 1]
        .groupby(["area", "area_title", "occ_std"], as_index=False)["a_median"]
        .mean()
        .rename(columns={"a_median": "wage_strand_avg"})
    )

    # Non-stranded industries
    non_avg = (
        df_wage[df_wage["is_strand"] == 0]
        .groupby(["area", "area_title", "occ_std"], as_index=False)["a_median"]
        .mean()
        .rename(columns={"a_median": "wage_non_avg"})
    )

    # --- 5.4 Deduplicate sim data ---
    sim_key = df_sim[
        ["occupation_code", "occupation_code_match", "occ_std", "occ_match_std"]
    ].drop_duplicates().copy()

    # --- 5.5 Left side: match occupation_code with stranded industry wages ---
    left_panel = sim_key.merge(
        strand_avg,
        on="occ_std",
        how="left"
    )

    # --- 5.6 Right side: match occupation_code_match with non-stranded industry wages ---
    right_panel = sim_key.merge(
        non_avg,
        left_on="occ_match_std",
        right_on="occ_std",
        how="left"
    ).rename(columns={
        "area":       "area_r",
        "area_title": "area_title_r"
    })

    # --- 5.7 Merge left and right by occupation pair + state ---
    panel = left_panel.merge(
        right_panel[[
            "occupation_code", "occupation_code_match",
            "area_r", "area_title_r", "wage_non_avg"
        ]],
        left_on=["occupation_code", "occupation_code_match", "area", "area_title"],
        right_on=["occupation_code", "occupation_code_match", "area_r", "area_title_r"],
        how="outer"
    )

    # Unify state variables
    panel["area_final"]       = panel["area"].fillna(panel["area_r"])
    panel["area_title_final"] = panel["area_title"].fillna(panel["area_title_r"])

    drop_cols = [c for c in ["area", "area_title", "area_r", "area_title_r"] if c in panel.columns]
    panel = panel.drop(columns=drop_cols).rename(columns={
        "area_final":       "area",
        "area_title_final": "area_title"
    })

    # --- 5.8 Calculate wage gap ---
    panel["wage_gap"] = panel["wage_non_avg"] - panel["wage_strand_avg"]

    # Reorder columns
    front_cols = [
        "occupation_code", "occupation_code_match",
        "occ_std", "occ_match_std",
        "area", "area_title",
        "wage_strand_avg", "wage_non_avg", "wage_gap"
    ]
    other_cols = [c for c in panel.columns if c not in front_cols]
    panel = panel[front_cols + other_cols]

    # --- 5.9 Export detailed data ---
    panel.to_csv(out_detail, index=False, encoding="utf-8-sig")

    # --- 5.10 Aggregate by occupation pair ---
    sum_df = (
        panel.groupby(["occupation_code", "occupation_code_match"], as_index=False)
        .agg(
            n_state_strand  =("wage_strand_avg", lambda x: x.notna().sum()),
            n_state_non     =("wage_non_avg",    lambda x: x.notna().sum()),
            n_state_both    =("wage_gap",        lambda x: x.notna().sum()),
            mean_wage_strand=("wage_strand_avg", "mean"),
            mean_wage_non   =("wage_non_avg",    "mean"),
            mean_wage_gap   =("wage_gap",        "mean")
        )
        .sort_values("mean_wage_gap", ascending=False)
        .reset_index(drop=True)
    )

    sum_df.to_csv(out_sum, index=False, encoding="utf-8-sig")

    # --- 5.11 Print results ---
    print(f"Detailed dataset rows: {len(panel):,}")
    print(f"Aggregated dataset rows: {len(sum_df):,}")
    print("\nTop 10 aggregated results:")
    print(sum_df.head(10).to_string(index=False))
    print(f"\nOutput files:\n  {out_detail}\n  {out_sum}\n")

print("=========== All tasks completed ===========")

In [ ]:
# Fig 5A Picture Drawing
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib import gridspec
from pathlib import Path

# =========================
# 1. File Paths
# =========================
file_path = r"Your path/Stranded_Occupations_Replication/Data/temp/sim1_pair_wage_detail.csv"

out_dir = Path(file_path).resolve().parent

out_png = out_dir / "wage_gap_lollipop_ABC_1.png"
out_pdf = out_dir / "wage_gap_lollipop_ABC_1.pdf"
out_svg = out_dir / "wage_gap_lollipop_ABC_1.svg"

# =========================
# 2. Read column names, auto-detect key variables
# =========================
header = pd.read_csv(file_path, nrows=0)
cols = [str(c).strip() for c in header.columns]

def find_col(columns, candidates):
    lower_map = {c.lower(): c for c in columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None

wage_gap_col = find_col(cols, ["wage_gap"])
wage_strand_col = find_col(cols, ["wage_strand_avg", "wage_strand", "strand_wage_avg"])
sim_col = find_col(cols, ["similarity", "skill_similarity", "sim", "match_similarity", "jaccard_similarity"])
# Find occupation_code_match column
occupation_code_match_col = find_col(cols, ["occupation_code_match", "occ_code_match", "match_code"])

if wage_gap_col is None:
    raise ValueError(f"wage_gap column not found. Current columns: {cols}")

if wage_strand_col is None:
    raise ValueError(f"wage_strand_avg column not found. Current columns: {cols}")

usecols = [wage_gap_col, wage_strand_col]
if sim_col is not None:
    usecols.append(sim_col)
if occupation_code_match_col is not None:
    usecols.append(occupation_code_match_col)

# =========================
# 3. Read only necessary columns for speed
# =========================
df = pd.read_csv(file_path, usecols=usecols)
df.columns = [str(c).strip() for c in df.columns]

# =========================
# 4. Data cleaning
#    Keep only rows with valid wage_gap values
# =========================
df[wage_gap_col] = pd.to_numeric(df[wage_gap_col], errors="coerce")
df[wage_strand_col] = pd.to_numeric(df[wage_strand_col], errors="coerce")

# Keep rows with valid values for both wage_gap and wage_strand_avg
df = df.dropna(subset=[wage_gap_col, wage_strand_col]).copy()

# Remove infinite values
df = df.replace([np.inf, -np.inf], np.nan)

# Exclude zero values for denominator
df = df[df[wage_strand_col] != 0].copy()

# Convert similarity column to numeric if exists
if sim_col is not None:
    df[sim_col] = pd.to_numeric(df[sim_col], errors="coerce")

# Remove samples where occupation_code_match == 475043
if occupation_code_match_col is not None:
    df[occupation_code_match_col] = pd.to_numeric(df[occupation_code_match_col], errors="coerce")
    n_before = len(df)
    df = df[df[occupation_code_match_col] != 475043].copy()
    n_after = len(df)
    print(f"Removed occupation_code_match==475043 samples: from {n_before} to {n_after}")

# X-axis variable
df["gap_ratio"] = df[wage_gap_col] / df[wage_strand_col]

# =========================
# 5. Percentage format
# =========================
def pct_fmt(x, pos):
    return f"{x:.0%}"

# =========================
# 6. Draw single panel
# =========================
def draw_rank_panel(ax_main, ax_bar, data, panel_label, title):
    data = data.sort_values("gap_ratio", ascending=True).reset_index(drop=True)
    n = len(data)

    if n == 0:
        ax_main.text(0.5, 0.5, "No observations", ha="center", va="center", fontsize=12)
        ax_main.set_axis_off()
        ax_bar.set_axis_off()
        return

    # Evenly space y-axis by rank position
    data["rank_y"] = (np.arange(1, n + 1) - 0.5) / n

    x = data["gap_ratio"].to_numpy()
    y = data["rank_y"].to_numpy()
    gap = data[wage_gap_col].to_numpy()

    # Lollipop line color matches point: green for negative, orange for positive/zero
    for xi, yi, gi in zip(x, y, gap):
        if gi < 0:
            line_color = "#1a9b2a"  # Green
        else:
            line_color = "#ff8c00"  # Orange
        ax_main.hlines(yi, 0, xi, linewidth=1.0, color=line_color, alpha=0.95, zorder=1)

    # Point colors: green for negative, orange for positive/zero
    point_colors = np.where(gap < 0, "#1a9b2a", "#ff8c00")
    ax_main.scatter(x, y, s=10, c=point_colors, edgecolors="none", zorder=3)

    # Zero reference line
    ax_main.axvline(0, linestyle=":", linewidth=1, color="gray", zorder=0)

    # X-axis range
    xmin = np.nanmin(x)
    xmax = np.nanmax(x)
    span = xmax - xmin
    if (not np.isfinite(span)) or (span == 0):
        span = max(abs(xmin), abs(xmax), 0.1)

    pad = span * 0.08
    ax_main.set_xlim(xmin - pad, xmax + pad)
    ax_main.set_ylim(0, 1.02)

    # Styling
    ax_main.xaxis.set_major_formatter(FuncFormatter(pct_fmt))
    ax_main.set_yticks([])
    ax_main.set_xlabel("")
    ax_main.grid(True, axis="both", linestyle=":", linewidth=0.8, alpha=0.7)

    ax_main.spines["top"].set_visible(False)
    ax_main.spines["right"].set_visible(False)
    ax_main.spines["left"].set_visible(False)

    ax_main.set_title(title, fontsize=14, weight="bold", pad=18)

    # Panel letter label
    ax_main.text(
        -0.14, 1.12, panel_label,
        transform=ax_main.transAxes,
        fontsize=20, fontweight="bold",
        bbox=dict(
            boxstyle="square,pad=0.18",
            facecolor="white",
            edgecolor="black",
            linewidth=1.2
        )
    )

    # =========================
    # Right side bar chart
    # Negative (green) at bottom, positive/zero (orange) at top
    # Percentages on left, arrows on right
    # =========================
    pos_share = (data[wage_gap_col] > 0).mean()
    neg_share = (data[wage_gap_col] < 0).mean()
    zero_share = (data[wage_gap_col] == 0).mean()

    pos_share_bar = pos_share + zero_share
    neg_share_bar = neg_share

    ax_bar.bar(0, neg_share_bar, bottom=0, width=0.20, color="#1a9b2a")
    ax_bar.bar(0, pos_share_bar, bottom=neg_share_bar, width=0.20, color="#ff8c00")

    ax_bar.set_xlim(-1.2, 1.2)
    ax_bar.set_ylim(0, 1)
    ax_bar.axis("off")

    # Percentage labels on left of bar
    ax_bar.text(
        -0.20, neg_share_bar / 2,
        f"{neg_share_bar:.0%}",
        va="center", ha="right", fontsize=12
    )
    ax_bar.text(
        -0.20, neg_share_bar + pos_share_bar / 2,
        f"{pos_share_bar:.0%}",
        va="center", ha="right", fontsize=12
    )

    # Arrows on right of bar
    ax_bar.text(
        0.28, neg_share_bar / 2,
        "↓",
        va="center", ha="left", fontsize=16
    )
    ax_bar.text(
        0.28, neg_share_bar + pos_share_bar / 2,
        "↑",
        va="center", ha="left", fontsize=16
    )

# =========================
# 7. Create A/B/C groups
# =========================
panels = [
    (
        "A",
        df.copy(),
        "Wages between fossil fuel occupations and their\nmatched occupations (similarity = 1)"
    )
]

# =========================
# 8. Create plot
# =========================
n_panels = len(panels)

fig = plt.figure(figsize=(7.5, 4.2 * n_panels), facecolor="white")
gs = gridspec.GridSpec(
    n_panels, 2,
    width_ratios=[15, 3],
    height_ratios=[1] * n_panels,
    wspace=0,
    hspace=0.55
)

for i, (label, subdf, ttl) in enumerate(panels):
    ax_main = fig.add_subplot(gs[i, 0], facecolor="white")
    ax_bar = fig.add_subplot(gs[i, 1], facecolor="white")
    draw_rank_panel(ax_main, ax_bar, subdf, label, ttl)

plt.tight_layout()

# =========================
# 9. Save files
# =========================
plt.savefig(out_png, dpi=400, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.savefig(out_pdf, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.savefig(out_svg, bbox_inches="tight", facecolor=fig.get_facecolor())

plt.show()

# =========================
# 10. Print summary statistics
# =========================
print("\n=== Panel summary ===")
for label, subdf, ttl in panels:
    tmp = subdf.copy()
    pos_share = (tmp[wage_gap_col] > 0).mean()
    neg_share = (tmp[wage_gap_col] < 0).mean()
    zero_share = (tmp[wage_gap_col] == 0).mean()
    print(
        f"{label}: n={len(tmp)}, "
        f"wage_gap>0 share={pos_share:.2%}, "
        f"wage_gap<0 share={neg_share:.2%}, "
        f"wage_gap=0 share={zero_share:.2%}"
    )

print("\nOutput files:")
print(out_png)
print(out_pdf)
print(out_svg)

In [ ]:
# Fig5B picture drawing
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib import gridspec
from pathlib import Path

# =========================
# 1. File path
# =========================
file_path = Path("Your path/Stranded_Occupations_Replication/Data/temp/sim851_pair_wage_detail.csv")
out_dir = file_path.resolve().parent

out_png = out_dir / "wage_gap_lollipop_ABC_851.png"
out_pdf = out_dir / "wage_gap_lollipop_ABC_851.pdf"
out_svg = out_dir / "wage_gap_lollipop_ABC_851.svg"

# =========================
# 2. Read column names, automatically identify key variables
# =========================
header = pd.read_csv(file_path, nrows=0)
cols = [str(c).strip() for c in header.columns]

def find_col(columns, candidates):
    lower_map = {c.lower(): c for c in columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None

wage_gap_col = find_col(cols, ["wage_gap"])
wage_strand_col = find_col(cols, ["wage_strand_avg", "wage_strand", "strand_wage_avg"])
sim_col = find_col(cols, ["similarity", "skill_similarity", "sim", "match_similarity", "jaccard_similarity"])
# New: look for occupation_code_match column
occupation_code_match_col = find_col(cols, ["occupation_code_match", "occ_code_match", "match_code"])

if wage_gap_col is None:
    raise ValueError(f"Column 'wage_gap' not found. Available columns: {cols}")

if wage_strand_col is None:
    raise ValueError(f"Column 'wage_strand_avg' not found. Available columns: {cols}")

usecols = [wage_gap_col, wage_strand_col]
if sim_col is not None:
    usecols.append(sim_col)
if occupation_code_match_col is not None:
    usecols.append(occupation_code_match_col)

# =========================
# 3. Read only necessary columns for speed
# =========================
df = pd.read_csv(file_path, usecols=usecols)
df.columns = [str(c).strip() for c in df.columns]

# =========================
# 4. Data cleaning
#    Keep only rows with non‑missing wage_gap
# =========================
df[wage_gap_col] = pd.to_numeric(df[wage_gap_col], errors="coerce")
df[wage_strand_col] = pd.to_numeric(df[wage_strand_col], errors="coerce")

# Keep only rows where both wage_gap and wage_strand_avg are numeric
df = df.dropna(subset=[wage_gap_col, wage_strand_col]).copy()

# Remove infinite values
df = df.replace([np.inf, -np.inf], np.nan)

# Denominator cannot be zero
df = df[df[wage_strand_col] != 0].copy()

# If similarity column exists, convert to numeric
if sim_col is not None:
    df[sim_col] = pd.to_numeric(df[sim_col], errors="coerce")

# [Modification 1] Remove rows where occupation_code_match == 475043
if occupation_code_match_col is not None:
    df[occupation_code_match_col] = pd.to_numeric(df[occupation_code_match_col], errors="coerce")
    n_before = len(df)
    df = df[df[occupation_code_match_col] != 475043].copy()
    n_after = len(df)
    print(f"Removed rows with occupation_code_match==475043: from {n_before} to {n_after}")

# Variable for x‑axis
df["gap_ratio"] = df[wage_gap_col] / df[wage_strand_col]

# =========================
# 5. Percentage formatter
# =========================
def pct_fmt(x, pos):
    return f"{x:.0%}"

# =========================
# 6. Draw a single panel
# =========================
def draw_rank_panel(ax_main, ax_bar, data, panel_label, title):
    data = data.sort_values("gap_ratio", ascending=True).reset_index(drop=True)
    n = len(data)

    if n == 0:
        ax_main.text(0.5, 0.5, "No observations", ha="center", va="center", fontsize=12)
        ax_main.set_axis_off()
        ax_bar.set_axis_off()
        return

    # y‑axis evenly spaced by rank
    data["rank_y"] = (np.arange(1, n + 1) - 0.5) / n

    x = data["gap_ratio"].to_numpy()
    y = data["rank_y"].to_numpy()
    gap = data[wage_gap_col].to_numpy()

    # [Modification 2] Lollipop stick colour matches point colour: negative = green, positive/zero = orange
    for xi, yi, gi in zip(x, y, gap):
        if gi < 0:
            line_color = "#1a9b2a"  # green
        else:
            line_color = "#ff8c00"  # orange
        ax_main.hlines(yi, 0, xi, linewidth=1.0, color=line_color, alpha=0.95, zorder=1)

    # 2) Point colours: negative = green, positive/zero = orange
    point_colors = np.where(gap < 0, "#1a9b2a", "#ff8c00")
    ax_main.scatter(x, y, s=10, c=point_colors, edgecolors="none", zorder=3)

    # Zero reference line
    ax_main.axvline(0, linestyle=":", linewidth=1, color="gray", zorder=0)

    # x‑axis limits
    xmin = np.nanmin(x)
    xmax = np.nanmax(x)
    span = xmax - xmin
    if (not np.isfinite(span)) or (span == 0):
        span = max(abs(xmin), abs(xmax), 0.1)

    pad = span * 0.08
    ax_main.set_xlim(xmin - pad, xmax + pad)
    ax_main.set_ylim(0, 1.02)

    # Styling
    ax_main.xaxis.set_major_formatter(FuncFormatter(pct_fmt))
    ax_main.set_yticks([])
    ax_main.set_xlabel("")   # no x‑axis label
    ax_main.grid(True, axis="both", linestyle=":", linewidth=0.8, alpha=0.7)

    ax_main.spines["top"].set_visible(False)
    ax_main.spines["right"].set_visible(False)
    ax_main.spines["left"].set_visible(False)

    ax_main.set_title(title, fontsize=14, weight="bold", pad=18)

    # Panel letter
    ax_main.text(
        -0.14, 1.12, panel_label,
        transform=ax_main.transAxes,
        fontsize=20, fontweight="bold",
        bbox=dict(
            boxstyle="square,pad=0.18",
            facecolor="white",
            edgecolor="black",
            linewidth=1.2
        )
    )

    # =========================
    # Right‑hand vertical bar
    # Negative (green) at bottom, positive/zero (orange) at top
    # Percentages on the left, arrows on the right
    # =========================
    pos_share = (data[wage_gap_col] > 0).mean()
    neg_share = (data[wage_gap_col] < 0).mean()
    zero_share = (data[wage_gap_col] == 0).mean()

    pos_share_bar = pos_share + zero_share
    neg_share_bar = neg_share

    ax_bar.bar(0, neg_share_bar, bottom=0, width=0.20, color="#1a9b2a")
    ax_bar.bar(0, pos_share_bar, bottom=neg_share_bar, width=0.20, color="#ff8c00")

    ax_bar.set_xlim(-1.2, 1.2)
    ax_bar.set_ylim(0, 1)
    ax_bar.axis("off")

    # Percentages on the left of the bar
    ax_bar.text(
        -0.20, neg_share_bar / 2,
        f"{neg_share_bar:.0%}",
        va="center", ha="right", fontsize=12
    )
    ax_bar.text(
        -0.20, neg_share_bar + pos_share_bar / 2,
        f"{pos_share_bar:.0%}",
        va="center", ha="right", fontsize=12
    )

    # Arrows on the right of the bar
    ax_bar.text(
        0.28, neg_share_bar / 2,
        "↓",
        va="center", ha="left", fontsize=16
    )
    ax_bar.text(
        0.28, neg_share_bar + pos_share_bar / 2,
        "↑",
        va="center", ha="left", fontsize=16
    )

# =========================
# 7. Build groups A / B / C
# =========================
panels = [
        (
            "B",
            df.copy(),
            "Wages between fossil fuel occupations and their\nmatched occupations (0.85 ≤ similarity < 1)"
        )
    ]

# =========================
# 8. Plot
# =========================
n_panels = len(panels)

fig = plt.figure(figsize=(7.5, 4.2 * n_panels), facecolor="white")
gs = gridspec.GridSpec(
    n_panels, 2,
    width_ratios=[15, 3],
    height_ratios=[1] * n_panels,
    wspace=0,
    hspace=0.55
)

for i, (label, subdf, ttl) in enumerate(panels):
    ax_main = fig.add_subplot(gs[i, 0], facecolor="white")
    ax_bar = fig.add_subplot(gs[i, 1], facecolor="white")
    draw_rank_panel(ax_main, ax_bar, subdf, label, ttl)

plt.tight_layout()

# =========================
# 9. Save
# =========================
plt.savefig(out_png, dpi=400, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.savefig(out_pdf, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.savefig(out_svg, bbox_inches="tight", facecolor=fig.get_facecolor())

plt.show()

# =========================
# 10. Output simple statistics
# =========================
print("\n=== Panel summary ===")
for label, subdf, ttl in panels:
    tmp = subdf.copy()
    pos_share = (tmp[wage_gap_col] > 0).mean()
    neg_share = (tmp[wage_gap_col] < 0).mean()
    zero_share = (tmp[wage_gap_col] == 0).mean()
    print(
        f"{label}: n={len(tmp)}, "
        f"wage_gap>0 proportion={pos_share:.2%}, "
        f"wage_gap<0 proportion={neg_share:.2%}, "
        f"wage_gap=0 proportion={zero_share:.2%}"
    )

print("\nOutput files:")
print(out_png)
print(out_pdf)
print(out_svg)

In [ ]:
# Fig 5C Picture Drawing
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib import gridspec
from pathlib import Path

# =========================
# 1. File Paths
# =========================
file_path = r"Your path/Stranded_Occupations_Replication/Data/temp/sim885_pair_wage_detail.csv"
out_dir = Path(file_path).resolve().parent

out_png = out_dir / "wage_gap_lollipop_ABC_885.png"
out_pdf = out_dir / "wage_gap_lollipop_ABC_885.pdf"
out_svg = out_dir / "wage_gap_lollipop_ABC_885.svg"

# =========================
# 2. Read column names, auto-detect key variables
# =========================
header = pd.read_csv(file_path, nrows=0)
cols = [str(c).strip() for c in header.columns]

def find_col(columns, candidates):
    lower_map = {c.lower(): c for c in columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None

wage_gap_col = find_col(cols, ["wage_gap"])
wage_strand_col = find_col(cols, ["wage_strand_avg", "wage_strand", "strand_wage_avg"])
sim_col = find_col(cols, ["similarity", "skill_similarity", "sim", "match_similarity", "jaccard_similarity"])
# New: Find occupation_code_match column
occupation_code_match_col = find_col(cols, ["occupation_code_match", "occ_code_match", "match_code"])

if wage_gap_col is None:
    raise ValueError(f"wage_gap column not found. Current columns: {cols}")

if wage_strand_col is None:
    raise ValueError(f"wage_strand_avg column not found. Current columns: {cols}")

usecols = [wage_gap_col, wage_strand_col]
if sim_col is not None:
    usecols.append(sim_col)
if occupation_code_match_col is not None:
    usecols.append(occupation_code_match_col)

# =========================
# 3. Read only necessary columns to improve speed
# =========================
df = pd.read_csv(file_path, usecols=usecols)
df.columns = [str(c).strip() for c in df.columns]

# =========================
# 4. Data cleaning
#    Keep only samples with valid wage_gap values
# =========================
df[wage_gap_col] = pd.to_numeric(df[wage_gap_col], errors="coerce")
df[wage_strand_col] = pd.to_numeric(df[wage_strand_col], errors="coerce")

# Keep rows with valid values for both wage_gap and wage_strand_avg
df = df.dropna(subset=[wage_gap_col, wage_strand_col]).copy()

# Remove infinite values
df = df.replace([np.inf, -np.inf], np.nan)

# Denominator cannot be zero
df = df[df[wage_strand_col] != 0].copy()

# Convert similarity column to numeric if it exists
if sim_col is not None:
    df[sim_col] = pd.to_numeric(df[sim_col], errors="coerce")

# Remove samples where occupation_code_match == 475043
if occupation_code_match_col is not None:
    df[occupation_code_match_col] = pd.to_numeric(df[occupation_code_match_col], errors="coerce")
    n_before = len(df)
    df = df[df[occupation_code_match_col] != 475043].copy()
    n_after = len(df)
    print(f"Removed occupation_code_match==475043 samples: from {n_before} to {n_after}")

# X-axis variable
df["gap_ratio"] = df[wage_gap_col] / df[wage_strand_col]

# =========================
# 5. Percentage format
# =========================
def pct_fmt(x, pos):
    return f"{x:.0%}"

# =========================
# 6. Draw single panel
# =========================
def draw_rank_panel(ax_main, ax_bar, data, panel_label, title):
    data = data.sort_values("gap_ratio", ascending=True).reset_index(drop=True)
    n = len(data)

    if n == 0:
        ax_main.text(0.5, 0.5, "No observations", ha="center", va="center", fontsize=12)
        ax_main.set_axis_off()
        ax_bar.set_axis_off()
        return

    # Evenly space y-axis by rank position
    data["rank_y"] = (np.arange(1, n + 1) - 0.5) / n

    x = data["gap_ratio"].to_numpy()
    y = data["rank_y"].to_numpy()
    gap = data[wage_gap_col].to_numpy()

    # Lollipop line color matches point: green for negative, orange for positive/zero
    for xi, yi, gi in zip(x, y, gap):
        if gi < 0:
            line_color = "#1a9b2a"  # Green
        else:
            line_color = "#ff8c00"  # Orange
        ax_main.hlines(yi, 0, xi, linewidth=1.0, color=line_color, alpha=0.95, zorder=1)

    # Point colors: green for negative, orange for positive/zero
    point_colors = np.where(gap < 0, "#1a9b2a", "#ff8c00")
    ax_main.scatter(x, y, s=10, c=point_colors, edgecolors="none", zorder=3)

    # Zero reference line
    ax_main.axvline(0, linestyle=":", linewidth=1, color="gray", zorder=0)

    # X-axis range
    xmin = np.nanmin(x)
    xmax = np.nanmax(x)
    span = xmax - xmin
    if (not np.isfinite(span)) or (span == 0):
        span = max(abs(xmin), abs(xmax), 0.1)

    pad = span * 0.08
    ax_main.set_xlim(xmin - pad, xmax + pad)
    ax_main.set_ylim(0, 1.02)

    # Styling
    ax_main.xaxis.set_major_formatter(FuncFormatter(pct_fmt))
    ax_main.set_yticks([])
    ax_main.set_xlabel("")   # Remove x-axis label
    ax_main.grid(True, axis="both", linestyle=":", linewidth=0.8, alpha=0.7)

    ax_main.spines["top"].set_visible(False)
    ax_main.spines["right"].set_visible(False)
    ax_main.spines["left"].set_visible(False)

    ax_main.set_title(title, fontsize=14, weight="bold", pad=18)

    # Panel letter label
    ax_main.text(
        -0.14, 1.12, panel_label,
        transform=ax_main.transAxes,
        fontsize=20, fontweight="bold",
        bbox=dict(
            boxstyle="square,pad=0.18",
            facecolor="white",
            edgecolor="black",
            linewidth=1.2
        )
    )

    # =========================
    # Right side vertical bar
    # Negative values at bottom (green), positive/zero at top (orange)
    # Values on left, arrows on right
    # =========================
    pos_share = (data[wage_gap_col] > 0).mean()
    neg_share = (data[wage_gap_col] < 0).mean()
    zero_share = (data[wage_gap_col] == 0).mean()

    pos_share_bar = pos_share + zero_share
    neg_share_bar = neg_share

    ax_bar.bar(0, neg_share_bar, bottom=0, width=0.20, color="#1a9b2a")
    ax_bar.bar(0, pos_share_bar, bottom=neg_share_bar, width=0.20, color="#ff8c00")

    ax_bar.set_xlim(-1.2, 1.2)
    ax_bar.set_ylim(0, 1)
    ax_bar.axis("off")

    # Percentage labels on left of bar
    ax_bar.text(
        -0.20, neg_share_bar / 2,
        f"{neg_share_bar:.0%}",
        va="center", ha="right", fontsize=12
    )
    ax_bar.text(
        -0.20, neg_share_bar + pos_share_bar / 2,
        f"{pos_share_bar:.0%}",
        va="center", ha="right", fontsize=12
    )

    # Arrows on right of bar
    ax_bar.text(
        0.28, neg_share_bar / 2,
        "↓",
        va="center", ha="left", fontsize=16
    )
    ax_bar.text(
        0.28, neg_share_bar + pos_share_bar / 2,
        "↑",
        va="center", ha="left", fontsize=16
    )

# =========================
# 7. Create A/B/C groups
# =========================
panels = [
        (
            "C",
            df.copy(),
            "Wages between fossil fuel occupations and their\nmatched occupations (0.8 ≤ similarity < 0.85)"
        )
    ]

# =========================
# 8. Create plot
# =========================
n_panels = len(panels)

fig = plt.figure(figsize=(7.5, 4.2 * n_panels), facecolor="white")
gs = gridspec.GridSpec(
    n_panels, 2,
    width_ratios=[15, 3],
    height_ratios=[1] * n_panels,
    wspace=0,
    hspace=0.55
)

for i, (label, subdf, ttl) in enumerate(panels):
    ax_main = fig.add_subplot(gs[i, 0], facecolor="white")
    ax_bar = fig.add_subplot(gs[i, 1], facecolor="white")
    draw_rank_panel(ax_main, ax_bar, subdf, label, ttl)

plt.tight_layout()

# =========================
# 9. Save files
# =========================
plt.savefig(out_png, dpi=400, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.savefig(out_pdf, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.savefig(out_svg, bbox_inches="tight", facecolor=fig.get_facecolor())

plt.show()

# =========================
# 10. Print summary statistics
# =========================
print("\n=== Panel summary ===")
for label, subdf, ttl in panels:
    tmp = subdf.copy()
    pos_share = (tmp[wage_gap_col] > 0).mean()
    neg_share = (tmp[wage_gap_col] < 0).mean()
    zero_share = (tmp[wage_gap_col] == 0).mean()
    print(
        f"{label}: n={len(tmp)}, "
        f"wage_gap>0 share={pos_share:.2%}, "
        f"wage_gap<0 share={neg_share:.2%}, "
        f"wage_gap=0 share={zero_share:.2%}"
    )

print("\nOutput files:")
print(out_png)
print(out_pdf)
print(out_svg)

In [ ]:
# Supplementary Fig 6 A-C for all panels picture drawing
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib import gridspec
from pathlib import Path

# =========================
# 1. Base Directory
# =========================
base_dir = Path(r"Your path/Stranded_Occupations_Replication/Data/temp")

# List of files to plot in batch
file_list = [
    "sim1_pair_wage_detail_14.csv",
    "sim1_pair_wage_detail_17.csv",
    "sim1_pair_wage_detail_20.csv",
    "sim851_pair_wage_detail_14.csv",
    "sim851_pair_wage_detail_17.csv",
    "sim851_pair_wage_detail_20.csv",
    "sim885_pair_wage_detail_14.csv",
    "sim885_pair_wage_detail_17.csv",
    "sim885_pair_wage_detail_20.csv",
]

# =========================
# 2. Helper Functions
# =========================
def find_col(columns, candidates):
    """Auto-detect column names by candidate list (case-insensitive)"""
    lower_map = {c.lower(): c for c in columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None

def pct_fmt(x, pos):
    """Format axis as percentage"""
    return f"{x:.0%}"

def infer_panel_info(file_name):
    """
    Determine panel letter, title, and output prefix from filename
    """
    stem = Path(file_name).stem

    if stem.startswith("sim1_"):
        panel_letter = "A"
        title = "Wages between fossil fuel occupations and their\nmatched occupations (similarity = 1)"
        out_prefix = "A"
    elif stem.startswith("sim851_"):
        panel_letter = "B"
        title = "Wages between fossil fuel occupations and their\nmatched occupations (0.85 ≤ similarity < 1)"
        out_prefix = "B"
    elif stem.startswith("sim885_"):
        panel_letter = "C"
        title = "Wages between fossil fuel occupations and their\nmatched occupations (0.8 ≤ similarity < 0.85)"
        out_prefix = "C"
    else:
        panel_letter = "A"
        title = "Wages between fossil fuel occupations and their\nmatched occupations"
        out_prefix = "X"

    # Extract year suffix: 14 / 17 / 20
    if stem.endswith("_14"):
        suffix = "14"
    elif stem.endswith("_17"):
        suffix = "17"
    elif stem.endswith("_20"):
        suffix = "20"
    else:
        suffix = ""

    if suffix:
        out_base = f"wage_gap_lollipop_{out_prefix}_{suffix}"
    else:
        out_base = f"wage_gap_lollipop_{out_prefix}"

    return panel_letter, title, out_base

def draw_rank_panel(ax_main, ax_bar, data, wage_gap_col, panel_label, title):
    """Draw single lollipop panel with side share bar"""
    data = data.sort_values("gap_ratio", ascending=True).reset_index(drop=True)
    n = len(data)

    if n == 0:
        ax_main.text(0.5, 0.5, "No observations", ha="center", va="center", fontsize=12)
        ax_main.set_axis_off()
        ax_bar.set_axis_off()
        return

    # Evenly space y-axis by rank position
    data["rank_y"] = (np.arange(1, n + 1) - 0.5) / n

    x = data["gap_ratio"].to_numpy()
    y = data["rank_y"].to_numpy()
    gap = data[wage_gap_col].to_numpy()

    # Lollipop lines: green for negative, orange for positive/zero
    for xi, yi, gi in zip(x, y, gap):
        if gi < 0:
            line_color = "#1a9b2a"  # Green
        else:
            line_color = "#ff8c00"  # Orange
        ax_main.hlines(yi, 0, xi, linewidth=0.8, color=line_color, alpha=0.95, zorder=1)

    # Point colors: green for negative, orange for positive/zero
    point_colors = np.where(gap < 0, "#1a9b2a", "#ff8c00")
    ax_main.scatter(x, y, s=6, c=point_colors, edgecolors="none", zorder=3)

    # Zero reference line
    ax_main.axvline(0, linestyle=":", linewidth=1, color="gray", zorder=0)

    # Set axis limits
    xmin = np.nanmin(x)
    xmax = np.nanmax(x)
    span = xmax - xmin
    if (not np.isfinite(span)) or (span == 0):
        span = max(abs(xmin), abs(xmax), 0.1)

    pad = span * 0.08
    ax_main.set_xlim(xmin - pad, xmax + pad)
    ax_main.set_ylim(0, 1.02)

    # Styling
    ax_main.xaxis.set_major_formatter(FuncFormatter(pct_fmt))
    ax_main.set_yticks([])
    ax_main.set_xlabel("")
    ax_main.grid(True, axis="both", linestyle=":", linewidth=0.8, alpha=0.7)

    ax_main.spines["top"].set_visible(False)
    ax_main.spines["right"].set_visible(False)
    ax_main.spines["left"].set_visible(False)

    ax_main.set_title(title, fontsize=12, weight="bold", pad=10)

    # Panel letter label (A/B/C)
    ax_main.text(
        -0.11, 1.06, panel_label,
        transform=ax_main.transAxes,
        fontsize=17, fontweight="bold",
        bbox=dict(
            boxstyle="square,pad=0.14",
            facecolor="white",
            edgecolor="black",
            linewidth=1.0
        )
    )

    # Right side share bar
    pos_share = (data[wage_gap_col] > 0).mean()
    neg_share = (data[wage_gap_col] < 0).mean()
    zero_share = (data[wage_gap_col] == 0).mean()

    pos_share_bar = pos_share + zero_share
    neg_share_bar = neg_share

    ax_bar.bar(0, neg_share_bar, bottom=0, width=0.12, color="#1a9b2a")
    ax_bar.bar(0, pos_share_bar, bottom=neg_share_bar, width=0.12, color="#ff8c00")

    ax_bar.set_xlim(-0.45, 0.45)
    ax_bar.set_ylim(0, 1)
    ax_bar.axis("off")

    # Percentage labels on left
    ax_bar.text(-0.08, neg_share_bar / 2, f"{neg_share_bar:.0%}",
                va="center", ha="right", fontsize=10)
    ax_bar.text(-0.08, neg_share_bar + pos_share_bar / 2, f"{pos_share_bar:.0%}",
                va="center", ha="right", fontsize=10)

    # Arrows on right
    ax_bar.text(0.08, neg_share_bar / 2, "↓",
                va="center", ha="left", fontsize=13)
    ax_bar.text(0.08, neg_share_bar + pos_share_bar / 2, "↑",
                va="center", ha="left", fontsize=13)

# =========================
# 3. Batch Generate All Plots
# =========================
summary_rows = []

for file_name in file_list:
    file_path = base_dir / file_name

    if not file_path.exists():
        print(f"Skipping: File not found -> {file_path}")
        continue

    print(f"Processing: {file_path}")

    # Read column names only
    header = pd.read_csv(file_path, nrows=0)
    cols = [str(c).strip() for c in header.columns]

    # Auto-detect key columns
    wage_gap_col = find_col(cols, ["wage_gap"])
    wage_strand_col = find_col(cols, ["wage_strand_avg", "wage_strand", "strand_wage_avg"])
    occupation_code_match_col = find_col(cols, ["occupation_code_match", "occ_code_match", "match_code"])

    if wage_gap_col is None or wage_strand_col is None:
        print(f"Skipping: Missing key columns -> {file_name}")
        continue

    # Read only necessary columns
    usecols = [wage_gap_col, wage_strand_col]
    if occupation_code_match_col is not None:
        usecols.append(occupation_code_match_col)

    df = pd.read_csv(file_path, usecols=usecols)
    df.columns = [str(c).strip() for c in df.columns]

    # Data cleaning
    df[wage_gap_col] = pd.to_numeric(df[wage_gap_col], errors="coerce")
    df[wage_strand_col] = pd.to_numeric(df[wage_strand_col], errors="coerce")

    df = df.dropna(subset=[wage_gap_col, wage_strand_col]).copy()
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df[df[wage_strand_col] != 0].copy()

    # Remove outliers: occupation_code_match == 475043
    if occupation_code_match_col is not None:
        df[occupation_code_match_col] = pd.to_numeric(df[occupation_code_match_col], errors="coerce")
        n_before = len(df)
        df = df[df[occupation_code_match_col] != 475043].copy()
        n_after = len(df)
        if n_before != n_after:
            print(f"  Removed occupation_code_match==475043: {n_before} -> {n_after}")

    if len(df) == 0:
        print(f"Skipping: No valid samples after cleaning -> {file_name}")
        continue

    # Compute x-axis variable
    df["gap_ratio"] = df[wage_gap_col] / df[wage_strand_col]

    # Get panel info from filename
    panel_letter, title, out_base = infer_panel_info(file_name)

    # Output paths
    out_png = base_dir / f"{out_base}.png"
    out_pdf = base_dir / f"{out_base}.pdf"
    out_svg = base_dir / f"{out_base}.svg"

    # Create figure
    fig = plt.figure(figsize=(7.5, 4.2), facecolor="white")
    gs = gridspec.GridSpec(
        1, 2,
        width_ratios=[15, 3],
        wspace=0,
        hspace=0.55
    )

    ax_main = fig.add_subplot(gs[0, 0], facecolor="white")
    ax_bar = fig.add_subplot(gs[0, 1], facecolor="white")

    # Draw panel
    draw_rank_panel(ax_main, ax_bar, df, wage_gap_col, panel_letter, title)

    # Save figures
    plt.tight_layout(pad=0.4)
    plt.savefig(out_png, dpi=400, bbox_inches="tight", facecolor="white")
    plt.savefig(out_pdf, bbox_inches="tight", facecolor="white")
    plt.savefig(out_svg, bbox_inches="tight", facecolor="white")
    plt.close(fig)

    # Calculate summary stats
    pos_share = (df[wage_gap_col] > 0).mean()
    neg_share = (df[wage_gap_col] < 0).mean()
    zero_share = (df[wage_gap_col] == 0).mean()

    # Save to summary
    summary_rows.append({
        "file_name": file_name,
        "panel": panel_letter,
        "n": len(df),
        "wage_gap_gt_0": pos_share,
        "wage_gap_lt_0": neg_share,
        "wage_gap_eq_0": zero_share,
        "png": str(out_png),
        "pdf": str(out_pdf),
        "svg": str(out_svg),
    })

# =========================
# 4. Export Summary Table
# =========================
summary_df = pd.DataFrame(summary_rows)
summary_out = base_dir / "wage_gap_lollipop_batch_summary.csv"
summary_df.to_csv(summary_out, index=False, encoding="utf-8-sig")

print("\nAll tasks completed.")
print(f"Summary file saved to: {summary_out}")
print(summary_df)

In [ ]:
# Fig 5D-F & Supplementary Fig 6 D-F for all panels picture drawing

"""
Functionality:
1. Vertically concatenate four groups of wage files:
   - Three files with _14 suffix
   - Three files with _17 suffix
   - Three files with _20 suffix
   - Three files with no suffix
2. Analyze each of the four merged datasets separately
3. Remove observations where occupation_code_match == 475043
4. Match occupation_information.xlsx type_skill_all by occupation_code
5. Perform paired t-tests between wage_strand_avg and wage_non_avg for four occupation types
6. Export one Excel result for each merged group
7. Draw one boxplot for each merged group, export PNG and SVG
8. Final outputs:
   4 Excel files
   4 boxplot PNG files
   4 boxplot SVG files
"""

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel
from matplotlib.patches import Patch

# =========================================================
# 1. Path Settings
# =========================================================
data_dir = Path("Your path/Stranded_Occupations_Replication/Data/temp")
occinfo_path = Path("Your path/Stranded_Occupations_Replication/Data/raw/occupation_information.xlsx")
out_dir = data_dir / "Fig5&spfig6_group"
out_dir.mkdir(parents=True, exist_ok=True)

# =========================================================
# 2. File Grouping
# =========================================================
group_files = {
    "14": [
        "sim1_pair_wage_detail_14.csv",
        "sim851_pair_wage_detail_14.csv",
        "sim885_pair_wage_detail_14.csv",
    ],
    "17": [
        "sim1_pair_wage_detail_17.csv",
        "sim851_pair_wage_detail_17.csv",
        "sim885_pair_wage_detail_17.csv",
    ],
    "20": [
        "sim1_pair_wage_detail_20.csv",
        "sim851_pair_wage_detail_20.csv",
        "sim885_pair_wage_detail_20.csv",
    ],
    "all": [
        "sim1_pair_wage_detail.csv",
        "sim851_pair_wage_detail.csv",
        "sim885_pair_wage_detail.csv",
    ],
}

skill_order = [
    "Non-Routine Cognitive",
    "Non-Routine Manual",
    "Routine Cognitive",
    "Routine Manual",
]

panel_titles = {
    "Non-Routine Cognitive": "NRC",
    "Non-Routine Manual": "NRM",
    "Routine Cognitive": "RC",
    "Routine Manual": "RM",
}

# =========================================================
# 3. Standardize occupation code format: convert to pure string codes
# =========================================================
def normalize_occ_code(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip()
    s = s.str.replace("-", "", regex=False)
    s = s.str.replace(" ", "", regex=False)
    s = s.str.replace(r"\.0+$", "", regex=True)
    s = s.replace({
        "": np.nan,
        "nan": np.nan,
        "None": np.nan,
        "<NA>": np.nan,
        "NA": np.nan
    })
    return s

# =========================================================
# 4. Read occupation information file
# =========================================================
occ = pd.read_excel(occinfo_path)
occ.columns = [c.strip() for c in occ.columns]

occ = occ[["occupation_code", "type_skill_all"]].copy()
occ["occupation_code"] = normalize_occ_code(occ["occupation_code"])
occ = occ[occ["type_skill_all"].isin(skill_order)].copy()
occ = occ.dropna(subset=["occupation_code", "type_skill_all"])
occ = occ.drop_duplicates(subset=["occupation_code"])

# =========================================================
# 5. Read and vertically concatenate one group
# =========================================================
def read_and_append_group(group_name: str, file_list: list[str]) -> pd.DataFrame:
    dfs = []

    for file_name in file_list:
        file_path = data_dir / file_name
        print(f"Reading [{group_name}]: {file_path}")

        df = pd.read_csv(file_path, low_memory=False)
        df.columns = [c.strip() for c in df.columns]

        required_cols = ["occupation_code", "occupation_code_match", "wage_strand_avg", "wage_non_avg"]
        for col in required_cols:
            if col not in df.columns:
                raise ValueError(f"Missing column in {file_name}: {col}")

        df = df.copy()
        df["source_file"] = file_name
        df["source_group"] = group_name

        df["occupation_code"] = normalize_occ_code(df["occupation_code"])
        df["occupation_code_match"] = normalize_occ_code(df["occupation_code_match"])
        df["wage_strand_avg"] = pd.to_numeric(df["wage_strand_avg"], errors="coerce")
        df["wage_non_avg"] = pd.to_numeric(df["wage_non_avg"], errors="coerce")

        df = df[df["occupation_code_match"] != "475043"].copy()
        df = df.dropna(subset=["occupation_code", "wage_strand_avg", "wage_non_avg"]).copy()

        dfs.append(df)

    return pd.concat(dfs, ignore_index=True)

# =========================================================
# 6. Paired t-test by skill type
# =========================================================
def paired_ttest_by_skill(df: pd.DataFrame, group_name: str) -> pd.DataFrame:
    results = []

    for skill in skill_order:
        sub = df[df["type_skill_all"] == skill].copy()
        sub = sub.dropna(subset=["wage_strand_avg", "wage_non_avg"])

        n = len(sub)

        if n == 0:
            results.append({
                "group_file": group_name,
                "type_skill_all": skill,
                "panel": panel_titles[skill],
                "N": 0,
                "mean_wage_strand_avg": np.nan,
                "mean_wage_non_avg": np.nan,
                "mean_diff_non_minus_strand": np.nan,
                "t_stat": np.nan,
                "p_value": np.nan,
            })
            continue

        t_stat, p_value = ttest_rel(
            sub["wage_non_avg"],
            sub["wage_strand_avg"],
            nan_policy="omit"
        )

        results.append({
            "group_file": group_name,
            "type_skill_all": skill,
            "panel": panel_titles[skill],
            "N": n,
            "mean_wage_strand_avg": sub["wage_strand_avg"].mean(),
            "mean_wage_non_avg": sub["wage_non_avg"].mean(),
            "mean_diff_non_minus_strand": (sub["wage_non_avg"] - sub["wage_strand_avg"]).mean(),
            "t_stat": t_stat,
            "p_value": p_value,
        })

    return pd.DataFrame(results)

# =========================================================
# 7. Draw 2x2 boxplot panel
# =========================================================
def draw_boxplot(df: pd.DataFrame, group_name: str, out_dir: Path) -> None:
    fig, axes = plt.subplots(2, 2, figsize=(7.2, 6.8), sharey=True)
    axes = axes.flatten()

    blue = "#1683f2"
    gray = "#a9a9a9"

    for i, skill in enumerate(skill_order):
        ax = axes[i]
        sub = df[df["type_skill_all"] == skill].copy()

        strand = sub["wage_strand_avg"].dropna().values
        non = sub["wage_non_avg"].dropna().values

        data_to_plot = [strand, non]

        bp = ax.boxplot(
            data_to_plot,
            patch_artist=True,
            widths=0.55,
            showfliers=False,
            medianprops=dict(color="white", linewidth=1.8),
            whiskerprops=dict(linewidth=1.5),
            capprops=dict(linewidth=1.5),
            boxprops=dict(linewidth=1.5),
        )

        bp["boxes"][0].set(facecolor=blue, edgecolor=blue)
        bp["boxes"][1].set(facecolor=gray, edgecolor=gray)

        bp["whiskers"][0].set(color=blue)
        bp["whiskers"][1].set(color=blue)
        bp["caps"][0].set(color=blue)
        bp["caps"][1].set(color=blue)

        bp["whiskers"][2].set(color=gray)
        bp["whiskers"][3].set(color=gray)
        bp["caps"][2].set(color=gray)
        bp["caps"][3].set(color=gray)

        ax.set_xticks([])
        ax.set_title(
        panel_titles[skill],
        fontsize=12,
        pad=4)

        ax.tick_params(axis="y", labelsize=10)
        ax.set_facecolor("white")

        for spine in ax.spines.values():
            spine.set_linewidth(1.0)
            spine.set_color("black")

        if i in [0, 2]:
            ax.set_ylabel("Annual wage ($)", fontsize=11)
        else:
            ax.set_ylabel("")

    # Main title
    fig.suptitle(
        f"Wages between fossil fuel occupations and their\n"
        f"matched occupations by occupational type",
        fontsize=17,
        fontweight="bold",
        y=0.97
    )

    legend_handles = [
        Patch(facecolor=blue, edgecolor=blue, label="Occupations in fossil fuel industries"),
        Patch(facecolor=gray, edgecolor=gray, label="Matched occupations in non-fossil fuel industries"),
    ]
    fig.legend(
        handles=legend_handles,
        loc="lower center",
        ncol=1,
        frameon=False,
        fontsize=10,
        bbox_to_anchor=(0.5, -0.01)
    )

    plt.tight_layout(rect=[0, 0.08, 1, 0.90])

    png_path = out_dir / f"boxplot_wage_by_skill_{group_name}.png"
    svg_path = out_dir / f"boxplot_wage_by_skill_{group_name}.svg"

    fig.savefig(png_path, dpi=400, bbox_inches="tight", facecolor="white")
    fig.savefig(svg_path, bbox_inches="tight", facecolor="white")
    plt.close(fig)

    print(f"Saved figure: {png_path}")
    print(f"Saved figure: {svg_path}")

# =========================================================
# 9. Main loop: Analyze four groups separately
# =========================================================
for group_name, file_list in group_files.items():
    print(f"\n========== Start group: {group_name} ==========")

    # 9.1 Concatenate group data
    df_group = read_and_append_group(group_name, file_list)

    # Save raw merged results
    merged_csv_path = out_dir / f"merged_wage_detail_{group_name}.csv"
    df_group.to_csv(merged_csv_path, index=False, encoding="utf-8-sig")
    print(f"Saved merged data: {merged_csv_path}")

    # 9.2 Match occupation skill types
    df_group = df_group.merge(
        occ,
        on="occupation_code",
        how="left"
    )

    # Keep only four occupation types
    df_group = df_group[df_group["type_skill_all"].isin(skill_order)].copy()

    # Set categorical order
    df_group["type_skill_all"] = pd.Categorical(
        df_group["type_skill_all"],
        categories=skill_order,
        ordered=True
    )

    # Save matched detailed data
    matched_csv_path = out_dir / f"merged_wage_detail_{group_name}_matched.csv"
    df_group.to_csv(matched_csv_path, index=False, encoding="utf-8-sig")
    print(f"Saved matched data: {matched_csv_path}")

    # 9.3 T-test results
    ttest_df = paired_ttest_by_skill(df_group, group_name)

    excel_path = out_dir / f"t_test_results_{group_name}.xlsx"
    with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
        ttest_df.to_excel(writer, index=False, sheet_name="t_test_results")

        # Add sample count table
        sample_count = (
            df_group.groupby("type_skill_all", observed=False)
            .size()
            .reset_index(name="N_rows")
        )
        sample_count.to_excel(writer, index=False, sheet_name="sample_count")

    print(f"Saved Excel: {excel_path}")

    # 9.4 Draw plot
    draw_boxplot(df_group, group_name, out_dir)

print("\nAll tasks completed.")